<a href="https://colab.research.google.com/github/mekdes10-30/Ev-parts-Task/blob/main/MEKDES_KIROS_G_10_TASK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EV Spare-Parts Import Prioritization  Research, Cleaning & Analysis Notebook

**Project question:** Which five EV replacement-part categories should a prospective importer prioritize for Ethiopia?

**Workflow:** assignment/scope → source and collection methodology → load raw files → inspect and validate → assess evidence quality and coverage → clean/standardize → validate cleaning → export cleaned data.

>

## 1. Assignment requirements and our decisions

1. Select **three** vehicles from the candidate list.
2. Investigate ten assigned part categories.
3. Build and clean a dataset from public online evidence.
4. Design an explainable scoring method.
5. Recommend five high-priority parts.

### Our documented scope decision
We deliberately selected **four vehicles**, rather than the PPT's three:
- BYD Seagull
- BYD Yuan Up
- Changan BenBen E-Star
- Toyota bZ4X

Why four?
- It gives us a broader range of vehicle models to investigate.
- It allows us to compare spare-part availability and compatibility across more manufacturers.
- It gives our research a wider evidence base for identifying potential import opportunities.

### Ten target categories
1. Brake pads
2. Brake discs
3. Cabin air filters
4. Wiper blades
5. Headlights
6. Tail lights
7. Side mirrors
8. Bumpers
9. Fenders/body panels
10. Shock absorbers



## 2. Research objective and interpretation

The project has two evidence dimensions:
- **International sourcing evidence:** part identification, supplier listings, price indications, compatibility claims and sourcing options.
- **Ethiopian market evidence:** local vehicle presence, local part availability/prices, supply gaps, and demand proxies.

International listings help investigate sourcing; they do **not** prove Ethiopian demand or local scarcity. A vehicle listing is not a registration or sales count. A supplier's capability statement is not an item-level stock confirmation.

## 3. Research Methodology and Sources

### Collection Approach

The raw dataset was collected in two phases:

- **Phase 1:** Initial research using publicly available online sources.
- **Phase 2:** Supplementary research to improve data coverage, particularly for BYD Yuan Up and Changan BenBen E-Star.

The research included web search results, fetched webpage content, supplier and catalogue pages, online marketplaces, Chinese-language searches, and Ethiopian market-context searches.

Each observed listing or evidence item was recorded separately and linked to its corresponding evidence-log entry. The collection distinguishes between information obtained from fetched webpages and information available only through search-result excerpts.

### Sources Used

The research used a combination of Ethiopian market sources and international technical and parts references:

- **Adey Market EV listings:** Ethiopian electric-vehicle market visibility.
- **Gezat Electric model catalogue:** Ethiopian EV and vehicle-model context.
- **Landin BYD Parts:** International BYD parts, fitment, and price references.
- **CQSW BYD Seagull part-number list:** International parts and part-number references.
- **Toyota bZ4X catalogue:** International vehicle and parts catalogue information.
- **NHTSA datasets and APIs:** Technical and vehicle-related information, rather than local market evidence.

The sources vary in geographic relevance and evidence strength. Not all are Ethiopian sources, and international references are not treated as proof of local availability.

### Source-Quality and Data-Handling Principles

The following principles guided the research and data preparation:

- Prefer identifiable manufacturers, suppliers, and established catalogues.
- Cross-check information using multiple independent sources where possible.
- Standardize currencies, units, and vehicle names.
- Record unavailable or unverified values instead of inventing information.
- Distinguish directly observed evidence from analytical assumptions and judgments.
- Preserve source URLs and collection dates to support traceability and reproducibility.

## 4. Our dataset design

The raw dataset contains 16 fields. The evidence log preserves observation-level provenance. We retain these schemas rather than silently changing the collection structure.

| Field | Why it is useful |
|---|---|
| record_id | Unique raw observation identifier |
| vehicle | Target vehicle |
| manufacturer | Vehicle/part manufacturer as recorded |
| part_category | One of the ten target categories (or contextual N/A row) |
| source_name | Source identity |
| source_type | Source class |
| source_url | Traceable source location |
| collection_date | When evidence was collected |
| evidence_type | Kind of observation |
| compatibility_status | Compatibility confidence/claim |
| compatibility_evidence | Basis for compatibility statement |
| part_number | Listing/catalogue part number |
| price | Observed price text/value |
| currency | Currency attached to price |
| availability_status | Availability claim as recorded |
| oe_part_number | Original-equipment part number if observed |

The evidence log records evidence ID, linked record ID, source, URL/date, observed information, page/product identifier, collection method, verification status, and access limitations.



## 5. Retrieve the original research files from GitHub

The four original research assets are retrieved directly from the public GitHub Release. The downloaded files are stored separately in `raw_original`; the original CSV contents are not modified.

The release tag is fixed to `v1.0-raw-data` so that the notebook uses a traceable research-data version.

In [43]:
# Retrieve the four original assets from the public GitHub Release
import requests
from pathlib import Path
import pandas as pd
import numpy as np
import csv
import re
import unicodedata
from IPython.display import display

BASE_DIR = Path("/content/ev_parts_project") if Path("/content").exists() else Path.cwd() / "ev_parts_project"
RAW_DIR = BASE_DIR / "raw_original"
OUTPUT_DIR = BASE_DIR / "output"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELEASE_API = "https://api.github.com/repos/mekdes10-30/Ev-parts-Task/releases/tags/v1.0-raw-data"
required_files = [
    "EV_Parts_Raw_Dataset.csv",
    "EV_Parts_Evidence_Log.csv",
    "EV_Parts_Coverage_Matrix.csv",
    "EV_Parts_Research_Notes.md",
]

response = requests.get(RELEASE_API, timeout=30)
response.raise_for_status()
release = response.json()
assets = {asset["name"]: asset["browser_download_url"] for asset in release.get("assets", [])}
missing_assets = [name for name in required_files if name not in assets]
if missing_assets:
    raise FileNotFoundError("Missing GitHub Release assets: " + ", ".join(missing_assets))

for filename in required_files:
    file_response = requests.get(assets[filename], timeout=60)
    file_response.raise_for_status()
    (RAW_DIR / filename).write_bytes(file_response.content)

print("Original release files retrieved:")
for filename in required_files:
    print(f"- {filename}")

Original release files retrieved:
- EV_Parts_Raw_Dataset.csv
- EV_Parts_Evidence_Log.csv
- EV_Parts_Coverage_Matrix.csv
- EV_Parts_Research_Notes.md


## 6. Load original files

Read every column as text initially. This prevents pandas from automatically converting identifiers, part numbers, dates, or strings that resemble numbers.

In [44]:
raw_file = next(RAW_DIR.rglob("EV_Parts_Raw_Dataset.csv"))
evidence_file = next(RAW_DIR.rglob("EV_Parts_Evidence_Log.csv"))
coverage_file = next(RAW_DIR.rglob("EV_Parts_Coverage_Matrix.csv"))
notes_file = next(RAW_DIR.rglob("EV_Parts_Research_Notes.md"))

raw_original = pd.read_csv(raw_file, encoding="utf-8", dtype=str, keep_default_na=False)
evidence_original = pd.read_csv(evidence_file, encoding="utf-8", dtype=str, keep_default_na=False)
coverage_original = pd.read_csv(coverage_file, encoding="utf-8", dtype=str, keep_default_na=False)

# Separate working copies. Do not modify *_original DataFrames.
raw = raw_original.copy(deep=True)
evidence = evidence_original.copy(deep=True)
coverage = coverage_original.copy(deep=True)

print("Raw:", raw.shape, "| Evidence:", evidence.shape, "| Coverage:", coverage.shape)
display(raw.head(5))

Raw: (197, 16) | Evidence: (197, 10) | Coverage: (40, 7)


,record_id,vehicle,manufacturer,part_category,source_name,source_type,source_url,collection_date,evidence_type,compatibility_status,compatibility_evidence,part_number,price,currency,availability_status,oe_part_number
0,REC-0001,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260F1,¥64,RMB,,EQEA-3501500CH
1,REC-0002,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260R1,¥64,RMB,,EQEA-3502140SS
2,REC-0003,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260F1P,¥64,RMB,,EQEA-3501500CH 14758996-00
3,REC-0004,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260R1P,¥64,RMB,,EQEA-3502140SS 14753311-00
4,REC-0005,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake discs,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Disc BYD0041960...",BYD0041960F1,¥108,RMB,,EM2E-3501011 13423467-00


## 7. Raw dataset inspection

Inspect dimensions, field names, data types, unique values, and blank strings. Blank strings are intentional representations of unavailable values in this collection.

In [45]:
print("Raw columns:")
print(raw.columns.tolist())
print("\nRaw dimensions:", raw.shape)
print("\nEvidence dimensions:", evidence.shape)
print("\nCoverage dimensions:", coverage.shape)

profile = pd.DataFrame({
    "column": raw.columns,
    "nonblank": [(raw[c].astype(str).str.strip() != "").sum() for c in raw.columns],
    "blank": [(raw[c].astype(str).str.strip() == "").sum() for c in raw.columns],
    "unique_values": [raw[c].nunique(dropna=False) for c in raw.columns]
})
display(profile)

Raw columns:
['record_id', 'vehicle', 'manufacturer', 'part_category', 'source_name', 'source_type', 'source_url', 'collection_date', 'evidence_type', 'compatibility_status', 'compatibility_evidence', 'part_number', 'price', 'currency', 'availability_status', 'oe_part_number']

Raw dimensions: (197, 16)

Evidence dimensions: (197, 10)

Coverage dimensions: (40, 7)


,column,nonblank,blank,unique_values
0,record_id,197,0,197
1,vehicle,197,0,5
2,manufacturer,197,0,17
3,part_category,197,0,13
4,source_name,197,0,56
5,source_type,197,0,38
6,source_url,197,0,128
7,collection_date,197,0,1
8,evidence_type,197,0,28
9,compatibility_status,197,0,13


## 8. Structural validation

Check the schema, identifier uniqueness, exact duplicate rows, and the one-to-one raw/evidence linkage. Any findings are displayed for review; this section does not delete records.

In [46]:
EXPECTED_RAW = [
    "record_id","vehicle","manufacturer","part_category","source_name","source_type",
    "source_url","collection_date","evidence_type","compatibility_status",
    "compatibility_evidence","part_number","price","currency","availability_status",
    "oe_part_number"
]
EXPECTED_EVIDENCE = [
    "evidence_id","record_id","source_name","source_url","collection_date",
    "observed_information","product_or_page_identifier","collection_method",
    "verification_status","access_limitations"
]

print("Missing raw columns:", sorted(set(EXPECTED_RAW)-set(raw.columns)))
print("Unexpected raw columns:", sorted(set(raw.columns)-set(EXPECTED_RAW)))
print("Missing evidence columns:", sorted(set(EXPECTED_EVIDENCE)-set(evidence.columns)))
print("Unexpected evidence columns:", sorted(set(evidence.columns)-set(EXPECTED_EVIDENCE)))

print("Raw row count:",len(raw),"unique record IDs:",raw.record_id.nunique())
print("Evidence row count:",len(evidence),"unique evidence IDs:",evidence.evidence_id.nunique())
print("Exact duplicate raw rows:",raw.duplicated().sum())
print("Exact duplicate evidence rows:",evidence.duplicated().sum())

raw_ids=set(raw.record_id)
ev_ids=set(evidence.record_id)
print("Raw IDs with no evidence:",len(raw_ids-ev_ids))
print("Evidence IDs not in raw:",len(ev_ids-raw_ids))

# Detect multiple evidence rows per raw record (expected one in this collection).
evidence_counts = evidence.groupby("record_id").size()

print("Records with exactly one evidence entry:",
      (evidence_counts == 1).sum())

print("Records with multiple evidence entries:",
      (evidence_counts > 1).sum())

Missing raw columns: []
Unexpected raw columns: []
Missing evidence columns: []
Unexpected evidence columns: []
Raw row count: 197 unique record IDs: 197
Evidence row count: 197 unique evidence IDs: 197
Exact duplicate raw rows: 0
Exact duplicate evidence rows: 0
Raw IDs with no evidence: 0
Evidence IDs not in raw: 0
Records with exactly one evidence entry: 197
Records with multiple evidence entries: 0


## 9. Coverage analysis (40 vehicle/category cells)

Coverage means that at least one observation was collected for a vehicle/category combination. It does **not** mean that compatibility, stock, demand, or evidence quality is confirmed.

The supplied matrix distinguishes model-named evidence from weak evidence and no evidence. We inspect it and independently derive the combinations represented in raw records.

In [47]:
display(coverage_original)

part_rows = raw[~raw["part_category"].str.strip().str.upper().str.startswith("N/A")].copy()
observed = (part_rows.groupby(["vehicle","part_category"]).size()
            .rename("raw_record_count").reset_index())
display(observed.sort_values(["vehicle","part_category"]))

print("Distinct vehicle/category pairs in raw part rows:",len(set(zip(observed.vehicle,observed.part_category))))

,vehicle,part_category,records_found,sources_investigated,evidence_found,research_limitations,additional_verification_required
0,BYD Seagull,Brake pads,8,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 8 from 4 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
1,BYD Seagull,Brake discs,3,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 3 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
2,BYD Seagull,Cabin air filters,6,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 6 from 5 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
3,BYD Seagull,Wiper blades,3,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 3 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
4,BYD Seagull,Headlights,5,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 5 from 3 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
5,BYD Seagull,Tail lights,4,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 4 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
6,BYD Seagull,Side mirrors,2,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 2 from 1 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
7,BYD Seagull,Bumpers,10,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 10 from 2 source name(s); status mix:...,Open Landin/JLL product detail pages; verify O...
8,BYD Seagull,Fenders/body panels,11,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 11 from 2 source name(s); status mix:...,Open Landin/JLL product detail pages; verify O...
9,BYD Seagull,Shock absorbers,7,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 7 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...


,vehicle,part_category,raw_record_count
0,BYD Seagull,Brake discs,3
1,BYD Seagull,Brake pads,8
2,BYD Seagull,Bumpers,10
3,BYD Seagull,Cabin air filters,6
4,BYD Seagull,Fenders/body panels,11
5,BYD Seagull,Headlights,5
6,BYD Seagull,Shock absorbers,7
7,BYD Seagull,Side mirrors,2
8,BYD Seagull,Tail lights,4
9,BYD Seagull,Wiper blades,3


Distinct vehicle/category pairs in raw part rows: 40


## 10. Evidence-log and provenance checks

The task requires specification for URLs and collection dates and cross-checking where possible. This audit checks whether provenance fields are populated and whether raw/evidence source details agree. A URL being present does not itself prove that a page was fully opened or independently verified.

In [48]:
# Join the evidence log to raw records for consistency checks.
audit = raw_original[["record_id","source_name","source_url","collection_date"]].merge(
    evidence_original[["record_id","source_name","source_url","collection_date","verification_status","access_limitations"]],
    on="record_id",suffixes=("_raw","_evidence"),how="outer",indicator=True
)
print(audit["_merge"].value_counts())
for field in ["source_name","source_url","collection_date"]:
    mismatch = audit[f"{field}_raw"] != audit[f"{field}_evidence"]
    print(field,"mismatches:",mismatch.sum())
display(audit[audit["_merge"]!="both"])
display(audit[(audit["source_url_raw"]!=audit["source_url_evidence"])])

_merge
both          197
left_only       0
right_only      0
Name: count, dtype: int64
source_name mismatches: 0
source_url mismatches: 0
collection_date mismatches: 0


,record_id,source_name_raw,source_url_raw,collection_date_raw,source_name_evidence,source_url_evidence,collection_date_evidence,verification_status,access_limitations,_merge


,record_id,source_name_raw,source_url_raw,collection_date_raw,source_name_evidence,source_url_evidence,collection_date_evidence,verification_status,access_limitations,_merge


## 10.1 Independent source cross-checking



This section compares records belonging to the same vehicle and part category.

We examine:
- Whether different sources report the same part number or OE number.
- Whether the sources represent genuinely different suppliers or related websites.
- Whether compatibility claims agree or contain uncertainty.
- Whether the evidence was obtained from an opened page or only a search-result snippet.

Known related sources are grouped into the same source family to avoid counting them as independent confirmations.

**Important:** Matching part numbers indicate potential corroboration of part identity. They do not independently prove vehicle compatibility, model-year fitment, availability or orderability. All candidate matches remain subject to manual review.

In [49]:
# PART 10.1 — Corrected independent source cross-checking

import re
import pandas as pd
from itertools import combinations
from IPython.display import display

# ---------------------------------------------------------
# 1. Prepare raw records and evidence log
# ---------------------------------------------------------

required_raw = [
    "record_id", "vehicle", "part_category",
    "source_name", "source_url",
    "part_number", "oe_part_number"
]

required_evidence = [
    "record_id", "verification_status",
    "collection_method", "access_limitations"
]

missing_raw = [c for c in required_raw if c not in raw_original.columns]
missing_evidence = [c for c in required_evidence if c not in evidence_original.columns]

if missing_raw or missing_evidence:
    raise ValueError(
        f"Missing raw columns: {missing_raw}\n"
        f"Missing evidence columns: {missing_evidence}"
    )

crosscheck_data = raw_original[required_raw].copy()

# Rename evidence columns to prevent collisions during merging.
evidence_for_crosscheck = evidence_original[required_evidence].rename(
    columns={
        "verification_status": "evidence_verification_status",
        "collection_method": "evidence_collection_method",
        "access_limitations": "evidence_access_limitations"
    }
)

crosscheck_data = crosscheck_data.merge(
    evidence_for_crosscheck,
    on="record_id",
    how="left",
    validate="one_to_one"
)

# ---------------------------------------------------------
# 2. Identify source families
# ---------------------------------------------------------

def identify_source_family(source_name):
    name = str(source_name).lower()

    # Known related Toyota retailer websites
    if "toyotapartsnow" in name or "toyotapartsdeal" in name:
        return "ToyotaPartsNow/ToyotaPartsDeal"

    # Other known source groups
    if "landin" in name:
        return "Landin"

    if "jll" in name:
        return "JLL Auto Parts"

    if "cqsw" in name or "chongqing shiwei" in name:
        return "Chongqing Shiwei"

    if "aliexpress" in name:
        return "AliExpress"

    if "amazon" in name:
        return "Amazon"

    if "byd-accessories.pages.dev" in name:
        return "BYD Accessories Aggregator"

    if "directfactoryparts" in name:
        return "DirectFactoryParts"

    if "neozero" in name:
        return "NeoZeroAuto"

    if "north park toyota" in name:
        return "North Park Toyota"

    if "nucar" in name:
        return "Nucar Toyota"

    if "conquest" in name:
        return "Conquest Auto"

    if "nunawading" in name:
        return "Nunawading Toyota"

    # Alibaba sellers are kept separate by their recorded source name.
    if "alibaba" in name:
        return str(source_name).strip()

    return str(source_name).strip()


crosscheck_data["source_family"] = (
    crosscheck_data["source_name"].apply(identify_source_family)
)

# ---------------------------------------------------------
# 3. Classify the type of evidence actually collected
# ---------------------------------------------------------

def classify_evidence(row):
    status = str(row["evidence_verification_status"]).lower()
    method = str(row["evidence_collection_method"]).lower()
    source = str(row["source_name"]).lower()

    if "snippet" in status or "web_search" in method:
        return "Search-result snippet"

    if "web_fetch" in method and "landin" in source:
        return "Fetched Landin catalogue listing"

    if "web_fetch" in method:
        return "Fetched page/listing text"

    return "Other / unclear collection method"


crosscheck_data["evidence_type"] = crosscheck_data.apply(
    classify_evidence, axis=1
)

# ---------------------------------------------------------
# 4. Extract identifiers from structured fields only
# ---------------------------------------------------------

def extract_identifiers(value):
    """
    Extract identifier-like tokens from structured part-number fields.
    Does not extract numbers from product descriptions or titles.
    """
    if pd.isna(value):
        return set()

    value = str(value).upper().strip()

    if not value:
        return set()

    # Split fields that contain multiple identifiers.
    pieces = re.split(r"[,;/|\n]+", value)

    identifiers = set()

    for piece in pieces:
        piece = piece.strip()

        # Keep reasonably long alphanumeric identifiers.
        tokens = re.findall(r"[A-Z0-9][A-Z0-9-]{4,}", piece)

        for token in tokens:
            normalized = re.sub(r"[^A-Z0-9]", "", token)

            if len(normalized) >= 5:
                identifiers.add(normalized)

    return identifiers


crosscheck_data["identifiers"] = crosscheck_data.apply(
    lambda row: (
        extract_identifiers(row["part_number"])
        | extract_identifiers(row["oe_part_number"])
    ),
    axis=1
)

# ---------------------------------------------------------
# 5. Compare records within the same vehicle/category
# ---------------------------------------------------------

candidate_rows = []

part_records = crosscheck_data[
    crosscheck_data["part_category"].astype(str).str.strip().ne("")
    & ~crosscheck_data["part_category"].astype(str).str.upper().str.startswith("N/A")
].copy()

for (vehicle, category), group in part_records.groupby(
    ["vehicle", "part_category"], dropna=False
):

    records = group.to_dict("records")

    for record_a, record_b in combinations(records, 2):

        shared_identifiers = (
            record_a["identifiers"] & record_b["identifiers"]
        )

        if not shared_identifiers:
            continue

        for identifier in sorted(shared_identifiers):

            family_a = record_a["source_family"]
            family_b = record_b["source_family"]

            same_family = family_a == family_b

            candidate_rows.append({
                "vehicle": vehicle,
                "part_category": category,
                "matching_identifier": identifier,

                "record_id_1": record_a["record_id"],
                "source_name_1": record_a["source_name"],
                "source_family_1": family_a,
                "evidence_type_1": record_a["evidence_type"],
                "verification_status_1": record_a["evidence_verification_status"],

                "record_id_2": record_b["record_id"],
                "source_name_2": record_b["source_name"],
                "source_family_2": family_b,
                "evidence_type_2": record_b["evidence_type"],
                "verification_status_2": record_b["evidence_verification_status"],

                "same_source_family": same_family,

                "source_relationship_assessment": (
                    "Same identified source family"
                    if same_family
                    else "Potentially distinct source families — independence not established"
                )
            })


crosscheck_matches = pd.DataFrame(candidate_rows)

# ---------------------------------------------------------
# 6. Build a corrected summary
# ---------------------------------------------------------

if not crosscheck_matches.empty:

    summary_rows = []

    for (vehicle, category), group in crosscheck_matches.groupby(
        ["vehicle", "part_category"]
    ):

        record_ids = set(
            group["record_id_1"].tolist()
            + group["record_id_2"].tolist()
        )

        involved_records = crosscheck_data[
            crosscheck_data["record_id"].isin(record_ids)
        ]

        distinct_families = set(
            group["source_family_1"].tolist()
            + group["source_family_2"].tolist()
        )

        independent_candidate_rows = group[
            group["same_source_family"] == False
        ]

        distinct_identifiers = group[
            "matching_identifier"
        ].nunique()

        summary_rows.append({
            "vehicle": vehicle,
            "part_category": category,

            "records_involved": len(record_ids),

            "distinct_source_names": involved_records["source_name"].nunique(),

            "distinct_source_families": len(distinct_families),

            "search_snippet_records": (
                involved_records["evidence_type"] == "Search-result snippet"
            ).sum(),

            "fetched_catalogue_records": (
                involved_records["evidence_type"] == "Fetched Landin catalogue listing"
            ).sum(),

            "other_fetched_page_records": (
                involved_records["evidence_type"] == "Fetched page/listing text"
            ).sum(),

            "pairwise_match_rows": len(group),

            "potentially_distinct_family_pair_rows": len(
                independent_candidate_rows
            ),

            "unique_matching_identifiers": distinct_identifiers,

            "review_status": (
                "Manual verification required — candidate matches only"
            )
        })

    crosscheck_summary = pd.DataFrame(summary_rows)

else:
    crosscheck_summary = pd.DataFrame()

# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

print("RAW RECORDS:", len(raw_original))
print("EVIDENCE RECORDS:", len(evidence_original))
print("PAIRWISE IDENTIFIER MATCH CANDIDATES:", len(crosscheck_matches))

if not crosscheck_matches.empty:
    print(
        "POTENTIALLY DISTINCT SOURCE-FAMILY PAIR ROWS:",
        crosscheck_matches[
            crosscheck_matches["same_source_family"] == False
        ].shape[0]
    )

print("\nCROSS-CHECK SUMMARY")
display(crosscheck_summary)

print("\nCANDIDATE MATCH DETAILS")
display(crosscheck_matches)

# ---------------------------------------------------------
# 8. Export reports
# ---------------------------------------------------------

summary_path = OUTPUT_DIR / "EV_Parts_Crosscheck_Summary.csv"
matches_path = OUTPUT_DIR / "EV_Parts_Crosscheck_Candidate_Matches.csv"

crosscheck_summary.to_csv(summary_path, index=False)
crosscheck_matches.to_csv(matches_path, index=False)

print("\nExported:")
print(summary_path)
print(matches_path)

RAW RECORDS: 197
EVIDENCE RECORDS: 197
PAIRWISE IDENTIFIER MATCH CANDIDATES: 42
POTENTIALLY DISTINCT SOURCE-FAMILY PAIR ROWS: 20

CROSS-CHECK SUMMARY


,vehicle,part_category,records_involved,distinct_source_names,distinct_source_families,search_snippet_records,fetched_catalogue_records,other_fetched_page_records,pairwise_match_rows,potentially_distinct_family_pair_rows,unique_matching_identifiers,review_status
0,BYD Seagull,Brake pads,4,1,1,0,4,0,2,0,2,Manual verification required — candidate match...
1,BYD Seagull,Bumpers,2,1,1,0,2,0,1,0,1,Manual verification required — candidate match...
2,BYD Seagull,Cabin air filters,3,3,3,3,0,0,4,4,2,Manual verification required — candidate match...
3,BYD Seagull,Headlights,2,2,2,1,1,0,1,1,1,Manual verification required — candidate match...
4,BYD Seagull,Shock absorbers,5,1,1,0,5,0,4,0,2,Manual verification required — candidate match...
5,BYD Yuan Up,Tail lights,3,3,3,3,0,0,2,2,2,Manual verification required — candidate match...
6,Toyota bZ4X,Brake discs,2,2,1,2,0,0,1,0,1,Manual verification required — candidate match...
7,Toyota bZ4X,Brake pads,8,4,3,8,0,0,13,7,2,Manual verification required — candidate match...
8,Toyota bZ4X,Bumpers,8,3,2,8,0,0,4,1,4,Manual verification required — candidate match...
9,Toyota bZ4X,Cabin air filters,6,4,3,6,0,0,6,4,2,Manual verification required — candidate match...



CANDIDATE MATCH DETAILS


,vehicle,part_category,matching_identifier,record_id_1,source_name_1,source_family_1,evidence_type_1,verification_status_1,record_id_2,source_name_2,source_family_2,evidence_type_2,verification_status_2,same_source_family,source_relationship_assessment
0,BYD Seagull,Brake pads,EQEA3501500CH,REC-0001,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,REC-0003,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,True,Same identified source family
1,BYD Seagull,Brake pads,EQEA3502140SS,REC-0002,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,REC-0004,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,True,Same identified source family
2,BYD Seagull,Bumpers,EQEA2804110A,REC-0020,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,REC-0021,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,True,Same identified source family
3,BYD Seagull,Cabin air filters,1476508400,REC-0051,byd-accessories.pages.dev ('Best BYD spare par...,BYD Accessories Aggregator,Search-result snippet,observed_in_search_snippet_page_not_opened; pa...,REC-0190,AliExpress item 3256807799561527,AliExpress,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — indepen...
4,BYD Seagull,Cabin air filters,1476508400,REC-0051,byd-accessories.pages.dev ('Best BYD spare par...,BYD Accessories Aggregator,Search-result snippet,observed_in_search_snippet_page_not_opened; pa...,REC-0191,Amazon.com search results 'seagull filter',Amazon,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — indepen...
5,BYD Seagull,Cabin air filters,1476508400,REC-0190,AliExpress item 3256807799561527,AliExpress,Search-result snippet,observed_in_search_snippet_page_not_opened,REC-0191,Amazon.com search results 'seagull filter',Amazon,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — indepen...
6,BYD Seagull,Cabin air filters,EQEA8100242,REC-0190,AliExpress item 3256807799561527,AliExpress,Search-result snippet,observed_in_search_snippet_page_not_opened,REC-0191,Amazon.com search results 'seagull filter',Amazon,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — indepen...
7,BYD Seagull,Headlights,EQEA4121010B,REC-0009,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,REC-0196,JLL Auto Parts - BYD Seagull category,JLL Auto Parts,Search-result snippet,observed_in_search_snippet_page_not_opened; nu...,False,Potentially distinct source families — indepen...
8,BYD Seagull,Shock absorbers,EQEA2905100A,REC-0036,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,REC-0039,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,True,Same identified source family
9,BYD Seagull,Shock absorbers,EQEA2905200A,REC-0037,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,REC-0040,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; com...,True,Same identified source family



Exported:
/content/ev_parts_project/output/EV_Parts_Crosscheck_Summary.csv
/content/ev_parts_project/output/EV_Parts_Crosscheck_Candidate_Matches.csv


In [50]:
# ============================================================
# PART 10.2 — MANUAL CROSS-CHECK REVIEW WORKSHEET
# ============================================================

review_candidates = [
    {
        "vehicle": "BYD Seagull",
        "part_category": "Cabin air filters",
        "matching_identifier": "1476508400",
        "record_ids": "REC-0051, REC-0190, REC-0191",
        "initial_observation": "Same identifier across an aggregator, AliExpress and Amazon search results.",
        "main_concern": "All are snippets; identifier inferred from aggregator title."
    },
    {
        "vehicle": "BYD Seagull",
        "part_category": "Cabin air filters",
        "matching_identifier": "EQEA8100242",
        "record_ids": "REC-0190, REC-0191",
        "initial_observation": "Same OE identifier appears in AliExpress and Amazon search results.",
        "main_concern": "Neither individual product page was opened."
    },
    {
        "vehicle": "BYD Seagull",
        "part_category": "Headlights",
        "matching_identifier": "EQEA4121010B",
        "record_ids": "REC-0009, REC-0196",
        "initial_observation": "Landin and JLL evidence contain the same OE identifier.",
        "main_concern": "JLL number-to-product mapping is inferred."
    },
    {
        "vehicle": "BYD Yuan Up",
        "part_category": "Tail lights",
        "matching_identifier": "SC3E4107300",
        "record_ids": "REC-0155, REC-0156",
        "initial_observation": "Same identifier appears in two Alibaba supplier records.",
        "main_concern": "Search snippets; exact product mapping and seller independence need checking."
    },
    {
        "vehicle": "BYD Yuan Up",
        "part_category": "Tail lights",
        "matching_identifier": "1463391400",
        "record_ids": "REC-0156, REC-0157",
        "initial_observation": "Same identifier appears in two Alibaba supplier records.",
        "main_concern": "Search snippets; product specification and compatibility need checking."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Brake pads",
        "matching_identifier": "0446578020",
        "record_ids": "REC-0055, REC-0058, REC-0060, REC-0061, REC-0062",
        "initial_observation": "Same identifier appears across three classified source families.",
        "main_concern": "Retailer relationship and conflicting vehicle fitment details."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Brake pads",
        "matching_identifier": "04466F4020",
        "record_ids": "REC-0056, REC-0057, REC-0059",
        "initial_observation": "Same identifier appears in multiple ToyotaPartsNow/ToyotaPartsDeal records.",
        "main_concern": "All records belong to the same identified source family."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Bumpers",
        "matching_identifier": "521194A957",
        "record_ids": "REC-0110, REC-0111",
        "initial_observation": "Same identifier appears in ToyotaPartsNow and North Park Toyota evidence.",
        "main_concern": "Exact bumper configuration and fitment need checking."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Cabin air filters",
        "matching_identifier": "8713942040",
        "record_ids": "REC-0073, REC-0075, REC-0078",
        "initial_observation": "Same identifier appears across Toyota retailer and Conquest Auto records.",
        "main_concern": "Confirm product description and exact vehicle applicability."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Cabin air filters",
        "matching_identifier": "8713942050",
        "record_ids": "REC-0074, REC-0076, REC-0077",
        "initial_observation": "Same identifier appears across Toyota retailer and Nucar Toyota records.",
        "main_concern": "Distinguish this filter variant from 87139-42040."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Wiper blades",
        "matching_identifier": "8521452110",
        "record_ids": "REC-0086, REC-0087",
        "initial_observation": "Same identifier appears in ToyotaPartsDeal and Nunawading Toyota evidence.",
        "main_concern": "Driver-side and right-hand descriptions may conflict."
    }
]

# Create the review DataFrame.
manual_review = pd.DataFrame(review_candidates)


# ------------------------------------------------------------
# MANUAL ASSESSMENTS
# ------------------------------------------------------------

assessments = [
    {
        "source_independence": "Potentially different channels; independence unverified",
        "product_mapping": "Partial match: identifier appears in three records, but its mapping comes from an aggregator title.",
        "vehicle_compatibility": "Seagull association is present in the collected evidence, but exact product fitment is not independently verified.",
        "final_classification": "Partially corroborated — unverified",
        "review_notes": (
            "REC-0051, REC-0190 and REC-0191 share identifier 1476508400. "
            "The records involve an aggregator, AliExpress and Amazon search results. "
            "All are snippet-level evidence. The aggregator-based identifier mapping "
            "and seller independence require further verification."
        )
    },
    {
        "source_independence": "Different platforms; seller independence unverified",
        "product_mapping": "Identifier matches across AliExpress and Amazon snippets; product mapping not independently inspected.",
        "vehicle_compatibility": "Seagull compatibility is suggested by the collected listing evidence but not confirmed through product pages or an official catalogue.",
        "final_classification": "Partially corroborated — unverified",
        "review_notes": (
            "Both records contain EQEA8100242. However, neither individual product "
            "page was opened. The same identifier across two marketplaces does not "
            "establish independent sellers or confirmed fitment."
        )
    },
    {
        "source_independence": "Different source names and domains; independence not confirmed",
        "product_mapping": "Identifier match observed; JLL's association of the number with a headlight is inferred.",
        "vehicle_compatibility": "Landin provides supplier-stated Seagull fitment; JLL does not independently establish the mapping and compatibility.",
        "final_classification": "Partially corroborated — mapping unverified",
        "review_notes": (
            "REC-0009 and REC-0196 share EQEA4121010B. Landin provides the Seagull "
            "headlight association, while JLL provides a matching identifier in a "
            "search result. The JLL product mapping needs confirmation."
        )
    },
    {
        "source_independence": "Two Alibaba supplier records; actual seller independence unverified",
        "product_mapping": "Matching identifier observed in both records; exact product specifications not independently checked.",
        "vehicle_compatibility": "Yuan Up is named in the collected listing evidence, but exact fitment remains supplier-stated and unverified.",
        "final_classification": "Partially corroborated — supplier evidence",
        "review_notes": (
            "REC-0155 and REC-0156 share SC3E4107300. The identifier appears across "
            "two Alibaba supplier records, but the evidence consists of search snippets. "
            "Seller independence, product mapping and exact vehicle fitment remain unconfirmed."
        )
    },
    {
        "source_independence": "Two Alibaba supplier records; actual seller independence unverified",
        "product_mapping": "Matching identifier observed; exact lamp specification and product variant not independently verified.",
        "vehicle_compatibility": "Yuan Up association is present in the collected supplier evidence, but compatibility has not been independently confirmed.",
        "final_classification": "Partially corroborated — supplier evidence",
        "review_notes": (
            "REC-0156 and REC-0157 share 1463391400. The identifier appears in two "
            "supplier records, but product specifications and seller independence "
            "have not been verified. Matching identifiers alone do not prove identical products."
        )
    },
    {
        "source_independence": "Three classified source families; some retailer relationships and independence remain uncertain",
        "product_mapping": "The same identifier 04465-78020 appears across five records. Product-number consistency is observed.",
        "vehicle_compatibility": "Conflicting supplier-stated fitment information exists. Exact bZ4X applicability requires official catalogue or VIN verification.",
        "final_classification": "Identifier corroborated — compatibility unresolved",
        "review_notes": (
            "REC-0055, REC-0058, REC-0060, REC-0061 and REC-0062 share 04465-78020. "
            "The identifier appears across ToyotaPartsNow, ToyotaPartsDeal, "
            "DirectFactoryParts and NeoZeroAuto. However, retailer relationships and "
            "conflicting fitment details prevent treating this as confirmed vehicle compatibility."
        )
    },
    {
        "source_independence": "Same identified source family: ToyotaPartsNow and ToyotaPartsDeal",
        "product_mapping": "Consistent identifier 04466-F4020 across the three records.",
        "vehicle_compatibility": "Supplier-stated bZ4X fitment exists, but no independent official EPC or VIN verification was performed.",
        "final_classification": "Same-source-family repetition",
        "review_notes": (
            "REC-0056, REC-0057 and REC-0059 contain the same identifier. The records "
            "belong to the same identified source family, so they should not be "
            "counted as independent supplier corroboration."
        )
    },
    {
        "source_independence": "Different retailer domains; independent supply chains not established",
        "product_mapping": "Matching identifier 52119-4A957 appears in both records and is categorized as a bumper part.",
        "vehicle_compatibility": "Supplier catalogue evidence associates the part with the bZ4X, but exact bumper configuration and fitment need confirmation.",
        "final_classification": "Partially corroborated — configuration unresolved",
        "review_notes": (
            "REC-0110 and REC-0111 contain the same identifier in ToyotaPartsNow and "
            "North Park Toyota evidence. The matching identifier supports a cross-retailer "
            "match, but the exact bumper configuration and independent supply relationship remain unverified."
        )
    },
    {
        "source_independence": "Different retailer domains; independent supply chains not established",
        "product_mapping": "Consistent cabin-filter identifier 87139-42040 across all three records.",
        "vehicle_compatibility": "Supplier-stated bZ4X fitment appears in the collected catalogue records; official vehicle-specific verification remains outstanding.",
        "final_classification": "Partially corroborated — supplier catalogue evidence",
        "review_notes": (
            "REC-0073, REC-0075 and REC-0078 contain the same identifier. ToyotaPartsNow, "
            "ToyotaPartsDeal and Conquest Auto provide matching catalogue evidence. "
            "This supports product-number consistency, but does not establish fully "
            "independent supply chains or VIN-level compatibility."
        )
    },
    {
        "source_independence": "Different retailer domains; independent supply chains not established",
        "product_mapping": "Consistent identifier 87139-42050 across the three records. It is a separate identifier from 87139-42040.",
        "vehicle_compatibility": "Supplier-stated bZ4X fitment appears in the collected records; official vehicle-specific verification remains outstanding.",
        "final_classification": "Partially corroborated — variant distinction retained",
        "review_notes": (
            "REC-0074, REC-0076 and REC-0077 share 87139-42050. The matching identifier "
            "is consistent across the records. It must remain separate from the "
            "87139-42040 filter group because the two identifiers represent different listed variants."
        )
    },
    {
        "source_independence": "Different retailer domains; independent supply chains not established",
        "product_mapping": "Matching identifier 85214-52110 appears in both records, but the exact wiper component and side description require clarification.",
        "vehicle_compatibility": "Supplier catalogue association with the bZ4X is present, but the driver-side/right-hand description conflict remains unresolved.",
        "final_classification": "Identifier match — component details unresolved",
        "review_notes": (
            "REC-0086 and REC-0087 contain 85214-52110 in ToyotaPartsDeal and "
            "Nunawading Toyota evidence. The matching identifier is useful, but "
            "conflicting side descriptions mean the exact component position should not be assumed."
        )
    }
]


# ------------------------------------------------------------
# ADD ASSESSMENTS TO WORKSHEET
# ------------------------------------------------------------

for column in assessments[0]:
    manual_review[column] = [
        assessment[column] for assessment in assessments
    ]


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("MANUAL CROSS-CHECK REVIEW WORKSHEET")
print("=" * 70)

print("Candidate identifier combinations:", len(manual_review))

print("\nFINAL CLASSIFICATION SUMMARY:")
display(
    manual_review["final_classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="number_of_candidates")
)

print("\nDETAILED MANUAL REVIEW:")
display(manual_review)


# ------------------------------------------------------------
# EXPORT COMPLETED WORKSHEET
# ------------------------------------------------------------

review_path = OUTPUT_DIR / "EV_Parts_Manual_Crosscheck_Review.csv"

manual_review.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nExported:", review_path)
print("Total reviewed candidates:", len(manual_review))
print("Unfilled assessment fields:",
      manual_review[
          [
              "source_independence",
              "product_mapping",
              "vehicle_compatibility",
              "final_classification",
              "review_notes"
          ]
      ].isna().sum().sum())

MANUAL CROSS-CHECK REVIEW WORKSHEET
Candidate identifier combinations: 11

FINAL CLASSIFICATION SUMMARY:


,classification,number_of_candidates
0,Partially corroborated — unverified,2
1,Partially corroborated — supplier evidence,2
2,Partially corroborated — mapping unverified,1
3,Identifier corroborated — compatibility unreso...,1
4,Same-source-family repetition,1
5,Partially corroborated — configuration unresolved,1
6,Partially corroborated — supplier catalogue ev...,1
7,Partially corroborated — variant distinction r...,1
8,Identifier match — component details unresolved,1



DETAILED MANUAL REVIEW:


,vehicle,part_category,matching_identifier,record_ids,initial_observation,main_concern,source_independence,product_mapping,vehicle_compatibility,final_classification,review_notes
0,BYD Seagull,Cabin air filters,1476508400,"REC-0051, REC-0190, REC-0191","Same identifier across an aggregator, AliExpre...",All are snippets; identifier inferred from agg...,Potentially different channels; independence u...,Partial match: identifier appears in three rec...,Seagull association is present in the collecte...,Partially corroborated — unverified,"REC-0051, REC-0190 and REC-0191 share identifi..."
1,BYD Seagull,Cabin air filters,EQEA8100242,"REC-0190, REC-0191",Same OE identifier appears in AliExpress and A...,Neither individual product page was opened.,Different platforms; seller independence unver...,Identifier matches across AliExpress and Amazo...,Seagull compatibility is suggested by the coll...,Partially corroborated — unverified,"Both records contain EQEA8100242. However, nei..."
2,BYD Seagull,Headlights,EQEA4121010B,"REC-0009, REC-0196",Landin and JLL evidence contain the same OE id...,JLL number-to-product mapping is inferred.,Different source names and domains; independen...,Identifier match observed; JLL's association o...,Landin provides supplier-stated Seagull fitmen...,Partially corroborated — mapping unverified,REC-0009 and REC-0196 share EQEA4121010B. Land...
3,BYD Yuan Up,Tail lights,SC3E4107300,"REC-0155, REC-0156",Same identifier appears in two Alibaba supplie...,Search snippets; exact product mapping and sel...,Two Alibaba supplier records; actual seller in...,Matching identifier observed in both records; ...,Yuan Up is named in the collected listing evid...,Partially corroborated — supplier evidence,REC-0155 and REC-0156 share SC3E4107300. The i...
4,BYD Yuan Up,Tail lights,1463391400,"REC-0156, REC-0157",Same identifier appears in two Alibaba supplie...,Search snippets; product specification and com...,Two Alibaba supplier records; actual seller in...,Matching identifier observed; exact lamp speci...,Yuan Up association is present in the collecte...,Partially corroborated — supplier evidence,REC-0156 and REC-0157 share 1463391400. The id...
5,Toyota bZ4X,Brake pads,0446578020,"REC-0055, REC-0058, REC-0060, REC-0061, REC-0062",Same identifier appears across three classifie...,Retailer relationship and conflicting vehicle ...,Three classified source families; some retaile...,The same identifier 04465-78020 appears across...,Conflicting supplier-stated fitment informatio...,Identifier corroborated — compatibility unreso...,"REC-0055, REC-0058, REC-0060, REC-0061 and REC..."
6,Toyota bZ4X,Brake pads,04466F4020,"REC-0056, REC-0057, REC-0059",Same identifier appears in multiple ToyotaPart...,All records belong to the same identified sour...,Same identified source family: ToyotaPartsNow ...,Consistent identifier 04466-F4020 across the t...,"Supplier-stated bZ4X fitment exists, but no in...",Same-source-family repetition,"REC-0056, REC-0057 and REC-0059 contain the sa..."
7,Toyota bZ4X,Bumpers,521194A957,"REC-0110, REC-0111",Same identifier appears in ToyotaPartsNow and ...,Exact bumper configuration and fitment need ch...,Different retailer domains; independent supply...,Matching identifier 52119-4A957 appears in bot...,Supplier catalogue evidence associates the par...,Partially corroborated — configuration unresolved,REC-0110 and REC-0111 contain the same identif...
8,Toyota bZ4X,Cabin air filters,8713942040,"REC-0073, REC-0075, REC-0078",Same identifier appears across Toyota retailer...,Confirm product description and exact vehicle ...,Different retailer domains; independent supply...,Consistent cabin-filter identifier 87139-42040...,Supplier-stated bZ4X fitment appears in the co...,Partially corroborated — supplier catalogue ev...,"REC-0073, REC-0075 and REC-0078 contain the sa..."
9,Toyota bZ4X,Cabin air filters,8713942050,"REC-0074, REC-0076, REC-0077",Same identifier appears across Toyota retailer...,Distinguish this 


Exported: /content/ev_parts_project/output/EV_Parts_Manual_Crosscheck_Review.csv
Total reviewed candidates: 11
Unfilled assessment fields: 0


## 11. Evidence quality and limitations

Create transparent review flags. These are **not scores** and are not grounds for deleting a row.

Evidence can be:
- model-named / direct or fetched information;
- search-result snippet only;
- title-level or multi-model claim;
- supplier capability statement;
- compatibility indicated but not confirmed;
- conflicting or uncertain.

The research notes describe important limitations: 12/39 covered combinations have weak-only evidence; Phase 2 records are search excerpts; supplier/operator independence is limited; and no part-level Ethiopian price/stock listing was found for the selected models.

In [51]:
# Simple field-level review flags.
raw["flag_missing_part_number"] = raw["part_number"].str.strip().eq("")
raw["flag_missing_oe_number"] = raw["oe_part_number"].str.strip().eq("")
raw["flag_missing_compatibility_evidence"] = raw["compatibility_evidence"].str.strip().eq("")
raw["flag_missing_price"] = raw["price"].str.strip().eq("")
raw["flag_missing_availability"] = raw["availability_status"].str.strip().eq("")
raw["flag_missing_source_url"] = raw["source_url"].str.strip().eq("")
raw["flag_compatibility_review"] = raw["compatibility_status"].str.lower().str.contains(
    "weak|uncertain|unconfirmed|multi|title|snippet|not confirmed|conflict",regex=True,na=False
)

flags = [c for c in raw.columns if c.startswith("flag_")]
display(raw[flags].sum().rename("records_flagged").to_frame())

# Source/evidence types as collected; inspect before any grouping decisions.
display(raw["evidence_type"].value_counts(dropna=False).to_frame("records"))
display(evidence["verification_status"].value_counts(dropna=False).to_frame("records"))

,records_flagged
flag_missing_part_number,64
flag_missing_oe_number,75
flag_missing_compatibility_evidence,0
flag_missing_price,72
flag_missing_availability,125
flag_missing_source_url,0
flag_compatibility_review,59


,records
evidence_type,
product listing (price listing),65
catalogue/product listing (price listing where price shown),41
product listing / catalogue entry,16
product listing title,8
product listing title (title names the category),8
supplier listing with category attributes,8
product listing,7
supplier capability statement (not an individual product listing),7
supplier capability statement,5


,records
verification_status,
observed_in_search_snippet_page_not_opened,137
observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (not independently verified),34
observed_directly_in_fetched_page_text,8
observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (not independently verified); category_mapping_inferred,7
observed_directly_in_fetched_page_text; category_mapping_inferred,4
observed_in_search_snippet_page_not_opened; Amharic copied exactly as returned (no normalisation),2
observed_in_search_snippet_page_not_opened; part_number_field_inferred_from_title_suffix,1
observed_in_search_snippet_page_not_opened; supplier_attribution_not_verified,1
observed_in_search_snippet_page_not_opened; Amharic text copied exactly as returned by the search tool,1


## 12. Cleaning and standardization

Cleaning is performed on a copy and is conservative:
- preserve every original column/value with a `_raw` suffix;
- trim leading/trailing whitespace;
- normalize Unicode text to NFC (canonical Unicode normalization);
- standardize vehicle/category labels only using explicit mappings;
- parse prices only when a numeric amount can be safely extracted, retaining original price and currency;
- do not infer missing stock, compatibility, price, or demand;
- do not merge listings merely because they look similar.

### Amharic handling
Original Amharic text is preserved without translation or spelling correction.
Unicode NFC normalization is applied to standardize canonically equivalent Unicode representations.


In [52]:
# Keep an untouched backup of the working raw values in *_raw columns.
for col in raw_original.columns:
    raw[f"{col}_raw"] = raw_original[col]

# Strip whitespace and NFC-normalize text in working columns only.
for col in raw_original.columns:
    raw[col] = raw[col].astype(str).str.strip().map(
        lambda x: unicodedata.normalize("NFC", x)
    )

# Explicit normalization maps; unknown labels remain as collected.
vehicle_map = {
    "BYD Seagull":"BYD Seagull",
    "BYD Yuan Up":"BYD Yuan Up",
    "Changan BenBen E-Star":"Changan BenBen E-Star",
    "Changan Benben E-Star":"Changan BenBen E-Star",
    "Toyota bZ4X":"Toyota bZ4X"
}
category_map = {
    "Brake pads":"Brake pads", "Brake discs":"Brake discs",
    "Cabin air filters":"Cabin air filters", "Cabin-air filters":"Cabin air filters",
    "Wiper blades":"Wiper blades", "Headlights":"Headlights",
    "Tail lights":"Tail lights", "Side mirrors":"Side mirrors",
    "Bumpers":"Bumpers", "Fenders/body panels":"Fenders/body panels",
    "Shock absorbers":"Shock absorbers"
}
raw["vehicle_clean"] = raw["vehicle"].replace(vehicle_map)
raw["part_category_clean"] = raw["part_category"].replace(category_map)

# Source-type normalization: conservative lowercase/trim only.
raw["source_type_clean"] = raw["source_type"].str.strip().str.lower()

display(raw[["vehicle","vehicle_clean"]].drop_duplicates())
display(raw[["part_category","part_category_clean"]].drop_duplicates())

,vehicle,vehicle_clean
0,BYD Seagull,BYD Seagull
49,BYD Yuan Up,BYD Yuan Up
53,Changan BenBen E-Star,Changan BenBen E-Star
54,Toyota bZ4X,Toyota bZ4X
138,General (not model-specific),General (not model-specific)


,part_category,part_category_clean
0,Brake pads,Brake pads
4,Brake discs,Brake discs
6,Wiper blades,Wiper blades
8,Headlights,Headlights
11,Tail lights,Tail lights
14,Side mirrors,Side mirrors
16,Bumpers,Bumpers
25,Fenders/body panels,Fenders/body panels
35,Shock absorbers,Shock absorbers
50,Cabin air filters,Cabin air filters


## 3. Price Parsing and Currency Conversion

Prices may be exact values, ranges, text descriptions, or price-on-request entries. Numeric helper columns are created to extract and classify available price information while preserving the original price and currency.

Eligible USD, CNY, and CAD prices are converted to ETB using a common National Bank of Ethiopia reference rate dated 16 November 2026. Unknown currencies and prices that cannot be reliably converted are left unconverted and flagged accordingly.

The conversion provides a standardized price reference; it does not represent the final Ethiopian landed cost, which may include shipping, customs duties, taxes, and other import expenses.

In [53]:
# ============================================================
# PART 13 — PRICE PARSING, CURRENCY STANDARDIZATION & ETB CONVERSION
# ============================================================

import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Common reference exchange rates
# Source: National Bank of Ethiopia (NBE)
# Reference date: 2026-11-16
# ------------------------------------------------------------

EXCHANGE_RATE_DATE = "2026-11-16"

EXCHANGE_RATES_TO_ETB = {
    "USD": 160.9987,
    "CNY": 24.2494,
    "CAD": 113.5305
}

# ------------------------------------------------------------
# 2. Standardize currency labels
# ------------------------------------------------------------

def standardize_currency(value):
    currency = str(value).strip().upper()

    if currency in ["RMB", "CNY"]:
        return "CNY"

    if currency == "CAD":
        return "CAD"

    if currency.startswith("USD"):
        return "USD"

    if currency in ["", "NAN", "NONE"]:
        return "UNKNOWN"

    return "UNKNOWN"


raw["currency_clean"] = raw["currency"].map(standardize_currency)

# ------------------------------------------------------------
# 3. Extract numerical values from original price expressions
# ------------------------------------------------------------

def extract_price_numbers(value):
    if not str(value).strip():
        return []

    s = str(value).replace(",", "")
    numbers = re.findall(r"\d+(?:\.\d+)?", s)

    return [float(n) for n in numbers]


raw["price_numbers"] = raw["price"].map(extract_price_numbers)

# Preserve compatibility with the previous notebook
raw["price_numeric_first"] = raw["price_numbers"].apply(
    lambda x: x[0] if len(x) > 0 else np.nan
)

# ------------------------------------------------------------
# 4. Interpret price expressions
# ------------------------------------------------------------

def classify_price(value, numbers):
    s = str(value).strip().lower()

    if not s:
        return "missing"

    if "request" in s:
        return "price_on_request"

    if len(numbers) == 0:
        return "unparseable"

    if len(numbers) >= 2:
        return "range_or_multiple_values_review"

    return "single_numeric_value"


raw["price_type"] = raw.apply(
    lambda row: classify_price(row["price"], row["price_numbers"]),
    axis=1
)

raw["price_parse_note"] = raw["price_type"]

# ------------------------------------------------------------
# 5. Create price bounds
# ------------------------------------------------------------

raw["price_numeric_low"] = raw["price_numbers"].apply(
    lambda x: min(x) if len(x) == 1 else np.nan
)

raw["price_numeric_high"] = raw["price_numbers"].apply(
    lambda x: max(x) if len(x) == 1 else np.nan
)

# Multiple-number expressions require manual review.

# ------------------------------------------------------------
# 6. Attach exchange rates
# ------------------------------------------------------------

raw["exchange_rate_to_etb"] = raw["currency_clean"].map(
    EXCHANGE_RATES_TO_ETB
)

raw["exchange_rate_date"] = EXCHANGE_RATE_DATE

# ------------------------------------------------------------
# 7. Convert prices into Ethiopian Birr
# ------------------------------------------------------------

raw["price_etb"] = (
    raw["price_numeric_first"] *
    raw["exchange_rate_to_etb"]
)

# Only convert single numerical prices with known currencies
raw.loc[
    raw["price_type"] != "single_numeric_value",
    "price_etb"
] = np.nan

# Round converted prices to two decimal places
raw["price_etb"] = raw["price_etb"].round(2)

# ------------------------------------------------------------
# 8. Conversion status
# ------------------------------------------------------------

def conversion_status(row):

    if row["price_type"] == "missing":
        return "missing_price"

    if row["price_type"] == "price_on_request":
        return "price_on_request"

    if row["price_type"] == "unparseable":
        return "unparseable_price"

    if row["currency_clean"] == "UNKNOWN":
        return "unknown_currency"

    if row["price_type"] != "single_numeric_value":
        return "manual_price_review"

    return "converted"


raw["conversion_status"] = raw.apply(
    conversion_status,
    axis=1
)

# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

display(
    raw[
        [
            "price",
            "currency",
            "currency_clean",
            "price_numeric_first",
            "price_type",
            "exchange_rate_to_etb",
            "price_etb",
            "conversion_status"
        ]
    ].head(25)
)

,price,currency,currency_clean,price_numeric_first,price_type,exchange_rate_to_etb,price_etb,conversion_status
0,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
1,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
2,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
3,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
4,¥108,RMB,CNY,108.0,single_numeric_value,24.2494,2618.94,converted
5,¥94,RMB,CNY,94.0,single_numeric_value,24.2494,2279.44,converted
6,Price on request,,UNKNOWN,NaN,price_on_request,NaN,NaN,price_on_request
7,Price on request,,UNKNOWN,NaN,price_on_request,NaN,NaN,price_on_request
8,¥418,RMB,CNY,418.0,single_numeric_value,24.2494,10136.25,converted
9,¥418,RMB,CNY,418.0,single_numeric_value,24.2494,10136.25,converted


In [54]:
# Currency conversion summary

print("Currency distribution:")
display(raw["currency_clean"].value_counts(dropna=False))

print("\nConversion status:")
display(raw["conversion_status"].value_counts(dropna=False))

print("\nConverted price statistics:")
display(raw.loc[raw["conversion_status"] == "converted", "price_etb"].describe())

print("\nUnknown currency records:")
display(
    raw.loc[
        raw["conversion_status"] == "unknown_currency",
        ["record_id", "vehicle", "part_category", "price", "currency"]
    ]
)

Currency distribution:


,count
currency_clean,
UNKNOWN,84
USD,82
CNY,29
CAD,2



Conversion status:


,count
conversion_status,
converted,97
missing_price,72
manual_price_review,16
price_on_request,12



Converted price statistics:


,price_etb
count,97.000000
mean,19029.856392
std,30891.912353
min,1246.130000
25%,4001.150000
50%,9336.020000
75%,22979.340000
max,177066.370000



Unknown currency records:


,record_id,vehicle,part_category,price,currency


In [55]:
# Inspect possible measurement information in text fields

measurement_terms = (
    r"\b\d+(?:\.\d+)?\s*"
    r"(?:mm|cm|m|inches|inch|in|kg|g|lbs?|"
    r"oz|pcs|pieces|sets|pairs)\b"
)

for col in [
    "compatibility_evidence",
    "part_number",
    "oe_part_number"
]:
    matches = raw[
        raw[col].str.contains(
            measurement_terms,
            case=False,
            regex=True,
            na=False
        )
    ]

    print(f"\n{col}: {len(matches)} possible measurement records")

    display(
        matches[["record_id", col]].head(20)
    )


compatibility_evidence: 7 possible measurement records


,record_id,compatibility_evidence
123,REC-0124,Vehicle/fitment text observed: Listed on '2023...
148,REC-0149,Listing title (Chinese): '万孚36028刹车片后片适配比亚迪秦EV...
164,REC-0165,Listing names 'BYD Atto 2 (3/2025+)' not 'Yuan...
174,REC-0175,Title: 'Applicable to the Front Bumper of Chan...
180,REC-0181,Excerpt: 'China Changan Auto Parts New Benben ...
190,REC-0191,Titles seen: 'Compatible With B-YD SEAGULL SEA...
191,REC-0192,Title: 'Cabin Air Filter Compatible For BYD Se...



part_number: 0 possible measurement records


,record_id,part_number



oe_part_number: 0 possible measurement records


,record_id,oe_part_number


## 14. Duplicate analysis

We distinguish:
- **Exact duplicate rows:** every field identical.
- **Potential same listing:** same source URL/product identifier or same vehicle/category/part number.
- **Different listings for the same part:** potentially useful price/supplier evidence, not automatically duplicates.
- **Variant differences:** may represent real fitment distinctions and must not be collapsed without evidence.

This section creates reports only; it does not remove records.

In [56]:
exact_dupes = raw_original[raw_original.duplicated(keep=False)].copy()
potential_key = ["vehicle_clean","part_category_clean","part_number","oe_part_number"]
potential_dupes = raw[raw.duplicated(subset=potential_key,keep=False) & raw["part_number"].ne("")].copy()

url_dupes = raw[raw.duplicated(subset=["source_url","vehicle_clean","part_category_clean"],keep=False) & raw["source_url"].ne("")].copy()

print("Exact duplicate raw rows:",len(exact_dupes))
print("Potential part-number matches:",len(potential_dupes))
print("Same URL/model/category groups:",len(url_dupes))
display(potential_dupes.sort_values(potential_key).head(40))

Exact duplicate raw rows: 0
Potential part-number matches: 30
Same URL/model/category groups: 64


,record_id,vehicle,manufacturer,part_category,source_name,source_type,source_url,collection_date,evidence_type,compatibility_status,...,price_numbers,price_numeric_first,price_type,price_parse_note,price_numeric_low,price_numeric_high,exchange_rate_to_etb,exchange_rate_date,price_etb,conversion_status
62,REC-0063,Toyota bZ4X,Toyota (genuine part number as displayed),Brake discs,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsnow.com/parts-select/20...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,[88.39],88.39,single_numeric_value,single_numeric_value,88.39,88.39,160.9987,2026-11-16,14230.68,converted
64,REC-0065,Toyota bZ4X,Toyota (genuine part number as displayed),Brake discs,toyotapartsdeal.com,genuine Toyota parts online retailer (US-orien...,https://toyotapartsdeal.com/oem-2023-toyota-bz...,2026-11-15,product listing / catalogue entry,supplier_stated_fitment_page_level_only,...,[],NaN,missing,missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
54,REC-0055,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsnow.com/parts-select/20...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,[91.83],91.83,single_numeric_value,single_numeric_value,91.83,91.83,160.9987,2026-11-16,14784.51,converted
57,REC-0058,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsdeal.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsdeal.com/oem-2024-toyot...,2026-11-15,product listing / catalogue entry,supplier_stated_fitment,...,[],NaN,missing,missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
59,REC-0060,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsdeal.com,genuine Toyota parts online retailer (US-orien...,https://toyotapartsdeal.com/oem-toyota-bz4x-br...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,[91.44],91.44,single_numeric_value,single_numeric_value,91.44,91.44,160.9987,2026-11-16,14721.72,converted
60,REC-0061,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,directfactoryparts.com,genuine OEM parts online retailer (US-oriented...,https://www.directfactoryparts.com/oem-parts/t...,2026-11-15,product listing (price listing),supplier_stated_fitment_conflicting,...,[92.99],92.99,single_numeric_value,single_numeric_value,92.99,92.99,160.9987,2026-11-16,14971.27,converted
61,REC-0062,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,neozeroauto.com,Toyota/Lexus parts specialist catalogue page (...,https://neozeroauto.com/part/04465-78020,2026-11-15,product listing / catalogue entry,supplier_stated_fitment_conflicting,...,[],NaN,missing,missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
55,REC-0056,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsnow.com/parts-select/20...,2026-11-15,product listing (price listing),supplier_stated_fitment_page_level_only,...,[55.09],55.09,single_numeric_value,single_numeric_value,55.09,55.09,160.9987,2026-11-16,8869.42,converted
56,REC-0057,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://toyotapartsnow.com/parts-select/toyota...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,[54.54],54.54,single_numeric_value,single_numeric_value,54.54,54.54,160.9987,2026-11-16,8780.87,converted
58,REC-0059,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsdeal.com,genuine Toyota parts online retailer (US-orien...,https://toyotapartsdeal.com/oem-toyota-bz4x-br...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,[51.75],51.75,single_numeric_value,single_numeric_value,51.75,51.75,160.9987,2026-11-16,8331.68,converted


## 15. Compatibility review

Compatibility is not assumed from a part name alone. The source's claim and the strength of evidence must remain visible. The raw compatibility status and evidence are retained; review flags do not upgrade or downgrade claims automatically.

In [57]:
compatibility_review = raw[[
    "record_id","vehicle_clean","part_category_clean","source_name","source_url",
    "compatibility_status","compatibility_evidence","part_number","oe_part_number",
    "flag_compatibility_review"
]].copy()
display(compatibility_review[compatibility_review["flag_compatibility_review"]].head(100))

,record_id,vehicle_clean,part_category_clean,source_name,source_url,compatibility_status,compatibility_evidence,part_number,oe_part_number,flag_compatibility_review
41,REC-0042,BYD Seagull,Brake pads,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,unconfirmed_supplier_general_claim,Page text: 'We support all the vehicle parts o...,,,True
42,REC-0043,BYD Seagull,Shock absorbers,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,unconfirmed_supplier_general_claim,Page text: 'We support all the vehicle parts o...,,,True
43,REC-0044,BYD Seagull,Bumpers,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,unconfirmed_supplier_general_claim,Page text: 'We support all the vehicle parts o...,,,True
44,REC-0045,BYD Seagull,Brake discs,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,unconfirmed_supplier_general_claim,Page's general inventory paragraph (not Seagul...,,,True
45,REC-0046,BYD Seagull,Headlights,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,unconfirmed_supplier_general_claim,Page's general inventory paragraph (not Seagul...,,,True
46,REC-0047,BYD Seagull,Tail lights,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,unconfirmed_supplier_general_claim,Page's general inventory paragraph (not Seagul...,,,True
47,REC-0048,BYD Seagull,Fenders/body panels,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,unconfirmed_supplier_general_claim,Page's general inventory paragraph (not Seagul...,,,True
48,REC-0049,BYD Seagull,Brake pads,byd-accessories.pages.dev ('Best BYD spare par...,https://byd-accessories.pages.dev/byd-spare-pa...,unconfirmed_multi_model_title,Listing title: 'Rear Brake Pad Set For BYD Sea...,,,True
49,REC-0050,BYD Yuan Up,Brake pads,byd-accessories.pages.dev ('Best BYD spare par...,https://byd-accessories.pages.dev/byd-spare-pa...,unconfirmed_multi_model_title,SAME underlying listing as the Seagull rear br...,,,True
50,REC-0051,BYD Seagull,Cabin air filters,byd-accessories.pages.dev ('Best BYD spare par...,https://byd-accessories.pages.dev/byd-spare-pa...,unconfirmed_listing_title_only,Listing title: 'Cabin A/C Fresh Air Filter For...,1476508400,,True


## 16. Cleaning validation and summary

Compare row counts before/after, check original fields remain, and review standardized labels. Cleaning should not silently drop observations.

In [58]:
print("Original raw rows:",len(raw_original))
print("Working rows:",len(raw))
print("Rows removed during cleaning:",len(raw_original)-len(raw))
print("Original columns preserved with *_raw:",all(f"{c}_raw" in raw.columns for c in raw_original.columns))

print("\nVehicle summary:")
display(raw[~raw["part_category_clean"].str.upper().str.startswith("N/A")].groupby("vehicle_clean").agg(
    records=("record_id","count"),categories=("part_category_clean","nunique")
).reset_index())

print("\nCategory summary:")
display(raw[~raw["part_category_clean"].str.upper().str.startswith("N/A")].groupby(
    "part_category_clean").size().rename("records").reset_index())

Original raw rows: 197
Working rows: 197
Rows removed during cleaning: 0
Original columns preserved with *_raw: True

Vehicle summary:


,vehicle_clean,records,categories
0,BYD Seagull,59,10
1,BYD Yuan Up,20,9
2,Changan BenBen E-Star,25,10
3,General (not model-specific),1,1
4,Toyota bZ4X,77,10



Category summary:


,part_category_clean,records
0,Brake discs,14
1,Brake pads,24
2,Brake pads (general market context; not EV-spe...,1
3,Bumpers,28
4,Cabin air filters,14
5,Fenders/body panels,18
6,Headlights,19
7,Shock absorbers,19
8,Side mirrors,15
9,Tail lights,14


## 17. Export cleaned working files

The original dataset remains unchanged. These files are explicitly named **Working** and retain original raw fields plus cleaning helpers.

In [59]:
cleaned_out = OUTPUT_DIR / "EV_Parts_Cleaned_Working.csv"
evidence_out = OUTPUT_DIR / "EV_Parts_Evidence_Log_Working.csv"
coverage_out = OUTPUT_DIR / "EV_Parts_Coverage_Matrix_Working.csv"
quality_out = OUTPUT_DIR / "EV_Parts_Data_Quality_Flags.csv"
duplicates_out = OUTPUT_DIR / "EV_Parts_Potential_Duplicates.csv"
compat_out = OUTPUT_DIR / "EV_Parts_Compatibility_Review.csv"

raw.to_csv(cleaned_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
evidence.to_csv(evidence_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
coverage.to_csv(coverage_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
raw[["record_id"]+flags].to_csv(quality_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
potential_dupes.to_csv(duplicates_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
compatibility_review.to_csv(compat_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)

print("Exported:")
for p in [cleaned_out,evidence_out,coverage_out,quality_out,duplicates_out,compat_out]:
    print(p)

Exported:
/content/ev_parts_project/output/EV_Parts_Cleaned_Working.csv
/content/ev_parts_project/output/EV_Parts_Evidence_Log_Working.csv
/content/ev_parts_project/output/EV_Parts_Coverage_Matrix_Working.csv
/content/ev_parts_project/output/EV_Parts_Data_Quality_Flags.csv
/content/ev_parts_project/output/EV_Parts_Potential_Duplicates.csv
/content/ev_parts_project/output/EV_Parts_Compatibility_Review.csv


In [60]:
from pathlib import Path

OUTPUT_DIR = Path("/content/ev_parts_project/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /content/ev_parts_project/output


In [61]:
import os

output_dir = "/content/ev_parts_project/output"

print("Files currently in output folder:")
for f in sorted(os.listdir(output_dir)):
    print(f)

Files currently in output folder:
EV_Parts_Category_Scores_Working.csv
EV_Parts_Cleaned_Working.csv
EV_Parts_Compatibility_Review.csv
EV_Parts_Coverage_Matrix_Working.csv
EV_Parts_Crosscheck_Candidate_Matches.csv
EV_Parts_Crosscheck_Summary.csv
EV_Parts_Data_Quality_Flags.csv
EV_Parts_Detailed_Evidence_Review.csv
EV_Parts_Evidence_Log_Working.csv
EV_Parts_Final_Top_Five_Recommendations.csv
EV_Parts_Manual_Crosscheck_Review.csv
EV_Parts_Potential_Duplicates.csv
EV_Parts_Sensitivity_Analysis.csv


## Part 18: Designing the Scoring Methodology

### 18.1

Our methodology uses evidence from the collected dataset to evaluate each vehicle–part combination.

### 18.2 Proposed Evaluation Factors

We propose five evaluation factors for determining inventory priority and assessing evidence reliability:

**1. Market Relevance**

Evaluates the available evidence indicating whether a vehicle and its replacement parts are relevant to the Ethiopian market.

Evidence may include vehicle-market observations, local marketplace listings, and other relevant market indicators.

These indicators are proxies for market relevance, not confirmed sales or actual replacement-part demand.

**2. Local Availability Gap**

Evaluates whether there is evidence that a replacement part is difficult to obtain locally.

Limited local listings or availability information may indicate a possible supply gap. However, a lack of online listings alone does not prove that a part is unavailable in Ethiopia.

**3. Compatibility Confidence**

Evaluates how strongly the collected evidence supports the compatibility of a replacement part with the specified vehicle.

Relevant evidence includes vehicle-specific catalogue entries, part numbers, OE references, and supplier fitment information.

A compatibility claim from a supplier is not automatically considered independently verified.

**4. Sourcing Practicality**

Evaluates how practical it appears to source the replacement part.

Relevant evidence includes identified suppliers, product listings, part identifiers, prices, and availability information.

Missing information about shipping, minimum order quantities, delivery time, and import costs will be acknowledged as a limitation.

**5. Evidence Confidence**

Evaluates the reliability and completeness of the collected evidence, considering source quality, evidence type, verification status, and independent corroboration.

Evidence confidence will be assessed separately from inventory priority because missing or weak evidence does not necessarily mean that a part has low demand or importance. It helps communicate how reliable the resulting assessment is.
For example, the absence of a BYD Yuan Up brake-disc listing in our investigated sources does not prove that the part is unnecessary or unavailable.

### 18.3 Scoring Unit

The primary scoring unit will be a vehicle–part combination.

With four vehicles and ten replacement-part categories, there are 40 possible combinations.

Each combination will be evaluated using the available evidence.

The results will then be aggregated to compare the ten part categories and identify the five recommended categories.

### 18.4 Methodological Principle

The scoring method will be evidence-based and transparent.

We will not invent demand figures, failure rates, local stock levels, or compatibility claims when the collected sources do not establish them.

The scoring scales, factor weights, missing-evidence treatment, and aggregation procedure will be defined in the following sections.

In [62]:
import pandas as pd

# Part 18: Proposed scoring framework

scoring_factors = pd.DataFrame([
    {
        "factor": "Market Relevance",
        "role": "Priority factor",
        "purpose": "Evaluate evidence of the vehicle's relevance to the Ethiopian market.",
        "supporting_evidence": (
            "Vehicle-market observations, local marketplace listings, "
            "and relevant market indicators"
        )
    },
    {
        "factor": "Local Availability Gap",
        "role": "Priority factor",
        "purpose": "Evaluate evidence of possible difficulty obtaining the part locally.",
        "supporting_evidence": (
            "Local listings, availability information, and observed supply gaps"
        )
    },
    {
        "factor": "Compatibility Confidence",
        "role": "Priority factor",
        "purpose": "Evaluate how strongly the evidence supports vehicle-part compatibility.",
        "supporting_evidence": (
            "Vehicle-specific catalogue entries, part numbers, OE references, "
            "and supplier fitment claims"
        )
    },
    {
        "factor": "Sourcing Practicality",
        "role": "Priority factor",
        "purpose": "Evaluate the practical possibility of sourcing the part.",
        "supporting_evidence": (
            "Supplier identification, product listings, prices, "
            "and availability information"
        )
    },
    {
        "factor": "Evidence Confidence",
        "role": "Separate reliability assessment",
        "purpose": "Communicate the strength and reliability of the evidence supporting the assessment.",
        "supporting_evidence": (
            "Source quality, verification status, corroboration, "
            "and completeness of supporting information"
        )
    }
])

display(scoring_factors)

,factor,role,purpose,supporting_evidence
0,Market Relevance,Priority factor,Evaluate evidence of the vehicle's relevance t...,"Vehicle-market observations, local marketplace..."
1,Local Availability Gap,Priority factor,Evaluate evidence of possible difficulty obtai...,"Local listings, availability information, and ..."
2,Compatibility Confidence,Priority factor,Evaluate how strongly the evidence supports ve...,"Vehicle-specific catalogue entries, part numbe..."
3,Sourcing Practicality,Priority factor,Evaluate the practical possibility of sourcing...,"Supplier identification, product listings, pri..."
4,Evidence Confidence,Separate reliability assessment,Communicate the strength and reliability of th...,"Source quality, verification status, corrobora..."


## Part 19: Scoring Framework

We will evaluate each of the 40 vehicle–part combinations using five factors:

| Factor | What we evaluate |
|---|---|
| Market Relevance | Relevance of the vehicle to the Ethiopian market. |
| Local Availability Gap | Evidence that the part is difficult to obtain locally. |
| Compatibility Confidence | Strength of the vehicle–part compatibility evidence. |
| Sourcing Practicality | Practicality of sourcing the part from suppliers. |
| Evidence Confidence | Reliability and strength of the supporting evidence. |

**Scoring approach:** Each factor will be scored on a scale of 1–5. The five scores will be combined using a weighted formula to calculate the final priority score.

The weights will reflect the relative importance of each factor in our research. Missing information will be treated as unknown rather than automatically assigned the lowest score.

In [63]:
# Part 19: Define the scoring scale

scoring_scale = pd.DataFrame({
    "score": [1, 2, 3, 4, 5],
    "interpretation": [
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
})

display(scoring_scale)

,score,interpretation
0,1,Very Low
1,2,Low
2,3,Moderate
3,4,High
4,5,Very High


### 19.1 Detailed Scoring Rubric

The following criteria guide how scores are assigned consistently across all vehicle–part combinations.

Scores must be supported by recorded evidence. Insufficient information will be marked **N/A** rather than guessed.

In [64]:
# Part 19.3: Detailed scoring rubric

scoring_rubric = {
    "Market Relevance": {
        1: "Very little evidence of vehicle presence in Ethiopia.",
        2: "Limited evidence from one weak source.",
        3: "Some credible evidence of local vehicle presence.",
        4: "Multiple credible indicators of market presence.",
        5: "Strong, consistent evidence from multiple sources."
    },

    "Local Availability Gap": {
        1: "Strong evidence that the part is readily available locally.",
        2: "Some evidence of local availability.",
        3: "Limited or inconsistent local availability evidence.",
        4: "Strong evidence of limited local availability.",
        5: "Multiple verified sources demonstrate a significant local supply gap."
    },

    "Compatibility Confidence": {
        1: "Generic listing with no clear vehicle compatibility.",
        2: "Ambiguous or multi-model fitment claim.",
        3: "Supplier-stated vehicle-specific fitment.",
        4: "Vehicle-specific catalogue entry with part identifier.",
        5: "Independently corroborated or officially verified fitment."
    },

    "Sourcing Practicality": {
        1: "No identifiable sourcing option.",
        2: "General supplier identified, but no specific product listing.",
        3: "Specific product listing and supplier identified.",
        4: "Product listing with price or useful availability information.",
        5: "Strong sourcing information, including confirmed stock and practical ordering details."
    },

    "Evidence Confidence": {
        1: "Weak, incomplete or ambiguous evidence.",
        2: "Single search-result snippet or poorly verified claim.",
        3: "Relevant source page or listing observed.",
        4: "Detailed evidence with strong source relevance.",
        5: "Reliable, independently corroborated or officially verified evidence."
    }
}

rubric_rows = []

for factor, levels in scoring_rubric.items():
    for score, description in levels.items():
        rubric_rows.append({
            "factor": factor,
            "score": score,
            "interpretation": scoring_scale.loc[
                scoring_scale["score"] == score, "interpretation"
            ].iloc[0],
            "criteria": description
        })

scoring_rubric_df = pd.DataFrame(rubric_rows)

display(scoring_rubric_df)

,factor,score,interpretation,criteria
0,Market Relevance,1,Very Low,Very little evidence of vehicle presence in Et...
1,Market Relevance,2,Low,Limited evidence from one weak source.
2,Market Relevance,3,Moderate,Some credible evidence of local vehicle presence.
3,Market Relevance,4,High,Multiple credible indicators of market presence.
4,Market Relevance,5,Very High,"Strong, consistent evidence from multiple sour..."
5,Local Availability Gap,1,Very Low,Strong evidence that the part is readily avail...
6,Local Availability Gap,2,Low,Some evidence of local availability.
7,Local Availability Gap,3,Moderate,Limited or inconsistent local availability evi...
8,Local Availability Gap,4,High,Strong evidence of limited local availability.
9,Local Availability Gap,5,Very High,Multiple verified sources demonstrate a signif...


## 19.2 Weight Assignment

The weights reflect the relative importance of each factor in determining import priority.

| Factor | Weight |
|---|---:|
| Market Relevance | 35% |
| Compatibility Confidence | 25% |
| Local Availability Gap | 20% |
| Sourcing Practicality | 15% |
| Evidence Confidence | 5% |
| **Total** | **100%** |

**Rationale:** Market relevance receives the highest weight because the objective is to identify parts associated with vehicles that matter in the Ethiopian market. Compatibility receives the second-highest weight because incorrect fitment creates financial and operational risks. Local availability and sourcing practicality determine whether an import opportunity is worthwhile and feasible. Evidence confidence receives a smaller weight because it evaluates the reliability of supporting information rather than the market opportunity itself.

In [65]:
# Part 19.2: Define and validate factor weights

factor_weights = {
    "Market Relevance": 0.35,
    "Compatibility Confidence": 0.25,
    "Local Availability Gap": 0.20,
    "Sourcing Practicality": 0.15,
    "Evidence Confidence": 0.05
}

weights_df = pd.DataFrame([
    {"factor": factor, "weight": weight}
    for factor, weight in factor_weights.items()
])

weights_df["weight_percentage"] = weights_df["weight"] * 100

display(weights_df[["factor", "weight_percentage"]])

total_weight = sum(factor_weights.values())

print(f"Total weight: {total_weight * 100:.0f}%")

assert abs(total_weight - 1.0) < 1e-9, "Weights must add up to 100%."

print("Weight validation passed.")

,factor,weight_percentage
0,Market Relevance,35.0
1,Compatibility Confidence,25.0
2,Local Availability Gap,20.0
3,Sourcing Practicality,15.0
4,Evidence Confidence,5.0


Total weight: 100%
Weight validation passed.


Part 20: Applying the Scoring Method


### 20.1 — Build the detailed evidence review

The 40 vehicle–part combinations in the coverage matrix are linked to their individual raw records. This creates a traceable evidence table containing source details, part identifiers, compatibility claims, prices, and availability information. The original raw dataset remains unchanged.

In [66]:
# Part 20.1: Build detailed evidence review for all 40 combinations

# Start with the original coverage matrix.
evaluation = coverage_original[["vehicle", "part_category"]].copy()

# Normalize coverage labels using the same mappings used during cleaning.
evaluation["vehicle_clean"] = (
    evaluation["vehicle"].astype(str).str.strip().replace(vehicle_map)
)
evaluation["part_category_clean"] = (
    evaluation["part_category"].astype(str).str.strip().replace(category_map)
)

# Identify valid part records from the cleaned working dataset.
part_records = raw[
    raw["part_category_clean"].ne("")
    & ~raw["part_category_clean"].str.upper().str.startswith("N/A")
].copy()

# Collect record IDs for each vehicle-category combination.
record_summary = (
    part_records
    .groupby(["vehicle_clean", "part_category_clean"])
    .agg(
        record_count=("record_id", "size"),
        record_ids=("record_id", lambda x: "; ".join(x.astype(str)))
    )
    .reset_index()
)

# Attach record counts and IDs to the evaluation table.
evaluation = evaluation.merge(
    record_summary,
    on=["vehicle_clean", "part_category_clean"],
    how="left",
    validate="one_to_one"
)
evaluation["record_count"] = evaluation["record_count"].fillna(0).astype(int)
evaluation["record_ids"] = evaluation["record_ids"].fillna("")

# Expand the record IDs into individual evidence rows.
evidence_review = evaluation[
    ["vehicle", "part_category", "record_ids"]
].copy()
evidence_review["record_id"] = evidence_review["record_ids"].apply(
    lambda x: x.split("; ") if x else []
)
evidence_review = evidence_review.drop(columns=["record_ids"]).explode("record_id")
evidence_review["record_id"] = evidence_review["record_id"].astype(str).str.strip()
evidence_review = evidence_review[
    evidence_review["record_id"].ne("") & evidence_review["record_id"].ne("nan")
]

# Attach details from the untouched original raw dataset.
evidence_review = evidence_review.merge(
    raw_original,
    on="record_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_raw")
)

# Select the original evidence fields available in this dataset.
preferred_fields = [
    "record_id", "vehicle", "part_category", "source_name", "source_url",
    "source_type", "evidence_type", "compatibility_status",
    "compatibility_evidence", "part_number", "oe_part_number", "price",
    "currency", "availability_status", "collection_date"
]
display_fields = [col for col in preferred_fields if col in evidence_review.columns]

print("Vehicle-category combinations:", len(evaluation))
print("Total linked evidence records:", len(evidence_review))
print("Evidence fields available:", display_fields)
display(evidence_review[display_fields])

evidence_review.to_csv(OUTPUT_DIR / "EV_Parts_Detailed_Evidence_Review.csv", index=False)
print("Detailed evidence review saved successfully.")

Vehicle-category combinations: 40
Total linked evidence records: 181
Evidence fields available: ['record_id', 'vehicle', 'part_category', 'source_name', 'source_url', 'source_type', 'evidence_type', 'compatibility_status', 'compatibility_evidence', 'part_number', 'oe_part_number', 'price', 'currency', 'availability_status', 'collection_date']


,record_id,vehicle,part_category,source_name,source_url,source_type,evidence_type,compatibility_status,compatibility_evidence,part_number,oe_part_number,price,currency,availability_status,collection_date
0,REC-0001,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd004...,aftermarket body/parts supplier catalogue (China),catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260F1,EQEA-3501500CH,¥64,RMB,,2026-11-15
1,REC-0002,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd004...,aftermarket body/parts supplier catalogue (China),catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260R1,EQEA-3502140SS,¥64,RMB,,2026-11-15
2,REC-0003,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd004...,aftermarket body/parts supplier catalogue (China),catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260F1P,EQEA-3501500CH 14758996-00,¥64,RMB,,2026-11-15
3,REC-0004,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd004...,aftermarket body/parts supplier catalogue (China),catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260R1P,EQEA-3502140SS 14753311-00,¥64,RMB,,2026-11-15
4,REC-0042,BYD Seagull,Brake pads,CQSW / Chongqing Shiwei Technology (cqswautopa...,https://www.cqswautoparts.com/wholesale-superi...,wholesale trading exporter website (China),supplier capability statement (not an individu...,unconfirmed_supplier_general_claim,Page text: 'We support all the vehicle parts o...,,,,,,2026-11-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,REC-0127,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyot...,genuine Toyota parts online retailer (US-orien...,product listing (price listing),supplier_stated_fitment,Vehicle/fitment text observed: Page lists vehi...,48520-80742,48520-80742,$128.69,USD,Ships in 1-3 Business Days,2026-11-15
177,REC-0128,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyot...,genuine Toyota parts online retailer (US-orien...,product listing (price listing),supplier_stated_fitment,Vehicle/fitment text observed: Page lists vehi...,48510-80C35,48510-80C35,$129.16,USD,Ships in 1-2 Business Days,2026-11-15
178,REC-0129,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyot...,genuine Toyota parts online retailer (US-orien...,product listing (price listing),supplier_stated_fitment,Vehicle/fitment text observed: Page lists vehi...,48520-80785,48520-80785,$124.15,USD,Ships in 1-2 Business Days,2026-11-15
179,REC-0130,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyot...,genuine Toyota parts online retailer (US-orien...,product listing (price listing),supplier_stated_fitment,Vehicle/fitment text observed: Page lists vehi...,48520-80738,48520-80738,$129.86,USD,,2026-11-15


Detailed evidence review saved successfully.


### 20.2 — Examine contextual market evidence

Records that do not describe a specific vehicle–part combination are examined separately. These may provide evidence of vehicle presence, local suppliers, importer activity, or broader market conditions. They are retained as context and are not treated as direct proof of item-level stock availability.

In [67]:
# Part 20.2: Examine contextual market evidence

part_record_ids = set(evidence_review["record_id"])
context_records = raw[~raw["record_id"].isin(part_record_ids)].copy()

print("Total contextual records:", len(context_records))

# Display the original evidence fields for readability; retain all columns in context_records.
context_fields = [
    "record_id", "vehicle", "part_category", "source_name", "source_type",
    "source_url", "evidence_type", "compatibility_status", "price",
    "currency", "availability_status", "collection_date"
]
display(context_records[[col for col in context_fields if col in context_records.columns]])

print("\nContextual records by source:")
if "source_name" in context_records.columns:
    display(
        context_records.groupby("source_name")
        .size()
        .reset_index(name="record_count")
        .sort_values("record_count", ascending=False)
    )

Total contextual records: 16


,record_id,vehicle,part_category,source_name,source_type,source_url,evidence_type,compatibility_status,price,currency,availability_status,collection_date
131,REC-0132,BYD Seagull,N/A (vehicle presence evidence - not a parts r...,Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/byd/seagull,market observation (vehicle presence),not_applicable_vehicle_presence,,,,2026-11-15
132,REC-0133,BYD Yuan Up,N/A (vehicle presence evidence - not a parts r...,Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/,market observation (vehicle presence),not_applicable_vehicle_presence,,,,2026-11-15
133,REC-0134,Changan BenBen E-Star,N/A (vehicle presence evidence - not a parts r...,Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/changan,market observation (vehicle presence),not_applicable_vehicle_presence,,,,2026-11-15
134,REC-0135,Toyota bZ4X,N/A (vehicle presence evidence - not a parts r...,Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/toyota,market observation (vehicle presence),not_applicable_vehicle_presence,,,,2026-11-15
135,REC-0136,BYD Yuan Up,N/A (vehicle presence evidence - not a parts r...,Hulu Market (hulumarket.com.et),Ethiopian classifieds marketplace,https://hulumarket.com.et/car/byd-yuan-plus,"vehicle sales listing (market observation, Amh...",not_applicable_vehicle_presence,,,,2026-11-15
136,REC-0137,Toyota bZ4X,N/A (vehicle presence evidence - not a parts r...,Beki Autocar (bekiautocar.com),Ethiopian car broker/dealer website,https://bekiautocar.com/,vehicle sales broker statement (market observa...,not_applicable_vehicle_presence,,,,2026-11-15
137,REC-0138,BYD Seagull,N/A (vehicle presence evidence - not a parts r...,Beki Autocar (bekiautocar.com),Ethiopian car broker/dealer website,https://bekiautocar.com/,vehicle sales broker statement (market observa...,not_applicable_vehicle_presence,,,,2026-11-15
138,REC-0139,General (not model-specific),N/A (market context - not one of the ten categ...,2merkato.com business directory - Ayu Begemdir...,Ethiopian business directory entry (local EV s...,https://www.2merkato.com/directory/41387-አዩ-በጌ...,market observation (local supplier self-descri...,not_applicable_market_context,,,,2026-11-15
139,REC-0140,General (not model-specific),N/A (market context - not one of the ten categ...,Addis Fortune - MOENCO introduces BYD Shark 6,Ethiopian news outlet,https://addisfortune.news/moenco-introduces-by...,market observation (news),not_applicable_market_context,,,,2026-11-15
140,REC-0141,General (not model-specific),N/A (market context - not one of the ten categ...,Mobility Rising newsletter - BYD signs partner...,industry newsletter,https://mobility-rising.beehiiv.com/p/byd-sign...,market observation (news),not_applicable_market_context,,,,2026-11-15



Contextual records by source:


,source_name,record_count
8,Gezat Electric (ev.gezat.et),4
4,Beki Autocar (bekiautocar.com),2
0,2merkato.com business directory - Ayu Begemdir...,1
1,Addis Fortune - MOENCO introduces BYD Shark 6,1
3,Adey Market (adey-market.com),1
2,Addis Fortune - spare-parts price article (Toy...,1
6,China-Global South Project - Ethiopia's middle...,1
5,CQSW case study - Ethiopian auto parts distrib...,1
7,Ethiopanorama (ethiopanorama.com) article,1
9,Hulu Market (hulumarket.com.et),1


### 20.3 — Assign scores to the five evaluation factors

Each vehicle–part combination is assessed using the five-factor scoring framework and the documented 1–5 scale. Compatibility, evidence confidence, and sourcing practicality are derived from the recorded evidence.

In [68]:
# ============================================================
# PART 20.3 — ASSIGN SCORES TO THE FIVE EVALUATION FACTORS
# ============================================================
%xmode Verbose
import numpy as np
import pandas as pd
import re

# ------------------------------------------------------------
# 0. PREPARATION
# ------------------------------------------------------------

PART_CATEGORIES = [
    "Brake pads",
    "Brake discs",
    "Cabin air filters",
    "Wiper blades",
    "Headlights",
    "Tail lights",
    "Side mirrors",
    "Bumpers",
    "Fenders/body panels",
    "Shock absorbers"
]

VEHICLES = [
    "BYD Seagull",
    "BYD Yuan Up",
    "Changan BenBen E-Star",
    "Toyota bZ4X"
]

def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


# ============================================================
# 1. MARKET RELEVANCE
# ============================================================
# Automatically evaluated using local contextual records.
# This measures evidence of vehicle presence in Ethiopia,


contextual_records = raw[
    ~raw["part_category"].isin(PART_CATEGORIES)
].copy()

market_assessments = []

for vehicle in VEHICLES:

    vehicle_records = contextual_records[
        contextual_records["vehicle"].astype(str).str.strip() == vehicle
    ].copy()

    distinct_sources = (
        vehicle_records["source_name"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .nunique()
    )

    if distinct_sources >= 3:
        score = 5
        reason = (
            "Strong evidence of vehicle presence from "
            "at least three distinct contextual sources."
        )

    elif distinct_sources >= 2:
        score = 4
        reason = (
            "Multiple contextual sources indicate local "
            "vehicle presence."
        )

    elif distinct_sources == 1:
        score = 3
        reason = (
            "Some credible evidence of local vehicle presence "
            "from one contextual source."
        )

    else:
        score = 1
        reason = "Very little evidence of vehicle presence in Ethiopia."

    market_assessments.append({
        "vehicle": vehicle,
        "Market Relevance": score,
        "market_evidence_sources": distinct_sources,
        "market_relevance_reason": reason
    })

market_assessment_df = pd.DataFrame(market_assessments)


# ============================================================
# 2. COMPATIBILITY CONFIDENCE
# ============================================================
# Uses the five levels defined in Part 19.3.

def score_compatibility(row):

    status = clean_text(row.get("compatibility_status"))
    evidence_type = clean_text(row.get("evidence_type"))
    verification = clean_text(row.get("verification_status"))

    part_number = clean_text(row.get("part_number"))
    oe_number = clean_text(row.get("oe_part_number"))

    has_identifier = bool(
        part_number not in ["", "nan", "none"]
        or oe_number not in ["", "nan", "none"]
    )

    # Level 5: Officially verified or independently corroborated
    if (
        "officially verified" in verification
        or "independently corroborated" in verification
    ):
        return 5

    # Level 2: Ambiguous or multi-model fitment
    if any(term in status + " " + evidence_type for term in [
        "multi_model",
        "multi-model",
        "title_only",
        "unconfirmed",
        "ambiguous"
    ]):
        return 2

    # Level 4: Vehicle-specific catalogue with an identifier
    if (
        "catalogue" in evidence_type
        and has_identifier
        and "supplier_stated_fitment" in status
    ):
        return 4

    # Level 3: Supplier-stated vehicle-specific fitment
    if "supplier_stated_fitment" in status:
        return 3

    # Level 1: Generic or unclear compatibility
    return 1


# ============================================================
# 3. EVIDENCE CONFIDENCE
# ============================================================
# Evaluates how reliable and complete the collected evidence is.

def score_evidence_confidence(row):

    evidence_type = clean_text(row.get("evidence_type"))
    verification = clean_text(row.get("verification_status"))

    # Level 5: Official or independently corroborated evidence
    if (
        "officially verified" in verification
        or "independently corroborated" in verification
    ):
        return 5

    # Level 2: Search-result snippets and title-only evidence
    if any(term in evidence_type + " " + verification for term in [
        "snippet",
        "search result",
        "search-result",
        "title only",
        "title-level"
    ]):
        return 2

    # Level 1: Generic, incomplete or ambiguous evidence
    if any(term in evidence_type for term in [
        "capability statement",
        "general claim",
        "market observation"
    ]):
        return 1

    # Level 4: Detailed catalogue or product evidence
    if (
        "catalogue" in evidence_type
        or "detailed" in verification
    ):
        return 4

    # Level 3: Relevant source page or listing observed
    if any(term in verification for term in [
        "fetched page",
        "fetched listing"
    ]):
        return 3

    if any(term in evidence_type for term in [
        "product listing",
        "supplier listing"
    ]):
        return 3

    return 1


# ============================================================
# 4. SOURCING PRACTICALITY
# ============================================================
# Evaluates the practical availability of a sourcing option.

def score_sourcing_practicality(row):

    evidence_type = clean_text(row.get("evidence_type"))
    price = clean_text(row.get("price"))
    availability = clean_text(row.get("availability_status"))

    verification = clean_text(row.get("verification_status"))

    has_price = price not in ["", "nan", "none", "price on request"]

    has_availability = availability not in ["", "nan", "none"]

    # Level 5: Confirmed stock and practical ordering details
    confirmed_stock = any(term in availability for term in [
        "in stock",
        "available now",
        "ready stock",
        "stock confirmed"
    ])

    ordering_details = any(term in availability for term in [
        "order",
        "shipping",
        "delivery",
        "minimum order"
    ])

    if confirmed_stock and ordering_details:
        return 5

    # Level 4: Product listing with price or useful availability info
    if (
        any(term in evidence_type for term in [
            "product listing",
            "catalogue",
            "supplier listing"
        ])
        and (has_price or has_availability)
    ):
        return 4

    # Level 3: Specific product listing and supplier identified
    if any(term in evidence_type for term in [
        "product listing",
        "catalogue",
        "supplier listing"
    ]):
        return 3

    # Level 2: General supplier identified
    if "supplier capability statement" in evidence_type:
        return 2

    # Level 1: No identifiable sourcing option
    return 1


# ============================================================
# # ============================================================
# 5. LOCAL AVAILABILITY GAP
# ============================================================

def score_local_availability_gap(vehicle, part_category):

    local_records = raw[
        (raw["vehicle"].astype(str).str.strip() == vehicle)
        &
        (raw["part_category"].astype(str).str.strip() == part_category)
    ].copy()

    # No matching vehicle-part records
    if local_records.empty:
        return pd.Series([
            3,
            "No specific local part-level availability evidence was found; "
            "the actual local supply gap remains uncertain."
        ])

    local_keywords = [
        "ethiopia",
        "ethiopian",
        "addis ababa",
        "local supplier",
        "local market"
    ]

    text_columns = [
        "source_name",
        "source_type",
        "evidence_type",
        "compatibility_evidence",
        "availability_status"
    ]

    # Keep only columns that actually exist
    text_columns = [
        column for column in text_columns
        if column in local_records.columns
    ]

    local_records["_local_text"] = local_records.apply(
        lambda row: " ".join(
            clean_text(row[column])
            for column in text_columns
        ),
        axis=1
    )

    local_records = local_records[
        local_records["_local_text"].apply(
            lambda text: any(
                keyword in text for keyword in local_keywords
            )
        )
    ]

    # No local evidence found: availability remains uncertain
    if local_records.empty:
        return pd.Series([
            3,
            "No specific local part-level availability evidence was found; "
            "the actual local supply gap remains uncertain."
        ])

    difficulty_keywords = [
        "unavailable",
        "difficult to source",
        "sourcing difficulty",
        "supply shortage",
        "import dependent",
        "import-dependent"
    ]

    difficulty_records = local_records[
        local_records["_local_text"].apply(
            lambda text: any(
                keyword in text for keyword in difficulty_keywords
            )
        )
    ]

    independent_sources = (
        difficulty_records["source_name"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", np.nan)
        .dropna()
        .nunique()
        if not difficulty_records.empty
        and "source_name" in difficulty_records.columns
        else 0
    )

    if independent_sources >= 2:
        return pd.Series([
            5,
            "Multiple local sources provide evidence of sourcing difficulty."
        ])

    if not difficulty_records.empty:
        return pd.Series([
            4,
            "Specific evidence indicates limited local availability."
        ])

    if len(local_records) >= 2:
        return pd.Series([
            1,
            "Multiple local records identify possible sourcing options."
        ])

    return pd.Series([
        2,
        "One local record identifies a possible sourcing option."
    ])

# ============================================================
# 6. SCORE INDIVIDUAL EVIDENCE RECORDS
# ============================================================

evidence_scores = evidence_review.copy()

evidence_scores["compatibility_score"] = evidence_scores.apply(
    score_compatibility,
    axis=1
)

evidence_scores["evidence_score"] = evidence_scores.apply(
    score_evidence_confidence,
    axis=1
)

evidence_scores["sourcing_score"] = evidence_scores.apply(
    score_sourcing_practicality,
    axis=1
)


# ============================================================
# 7. SUMMARIZE EVIDENCE BY VEHICLE-PART COMBINATION
# ============================================================

evidence_summary = (
    evidence_scores
    .groupby(["vehicle", "part_category"], as_index=False)
    .agg(
        compatibility_score=("compatibility_score", "max"),
        evidence_score=("evidence_score", "max"),
        sourcing_score=("sourcing_score", "max")
    )
)


# ============================================================
# 8. CREATE THE 40-COMBINATION SCORING TABLE
# ============================================================

scoring_table = evaluation[
    ["vehicle", "part_category", "record_count"]
].copy()

scoring_table = scoring_table.merge(
    evidence_summary,
    on=["vehicle", "part_category"],
    how="left"
)

# Market Relevance
scoring_table = scoring_table.merge(
    market_assessment_df,
    on="vehicle",
    how="left"
)

scoring_table["Compatibility Confidence"] = (
    scoring_table["compatibility_score"]
)

scoring_table["Sourcing Practicality"] = (
    scoring_table["sourcing_score"]
)

scoring_table["Evidence Confidence"] = (
    scoring_table["evidence_score"]
)



# ============================================================
# 9. ASSIGN LOCAL AVAILABILITY GAP SCORES AND REASONS
# ============================================================

local_gap_results = scoring_table.apply(
    lambda row: score_local_availability_gap(
        row["vehicle"],
        row["part_category"]
    ),
    axis=1
)

scoring_table["Local Availability Gap"] = local_gap_results[0]
scoring_table["local_gap_reason"] = local_gap_results[1]

# ============================================================
# 10. DISPLAY FINAL FACTOR SCORES
# ============================================================


factor_columns = [
    "Market Relevance",
    "Compatibility Confidence",
    "Local Availability Gap",
    "Sourcing Practicality",
    "Evidence Confidence"
]

display(
    scoring_table[
        ["vehicle", "part_category", "record_count"] + factor_columns
    ]
)

print("Total vehicle-part combinations:", len(scoring_table))
print("Scoring completed successfully.")

Exception reporting mode: Verbose


,vehicle,part_category,record_count,Market Relevance,Compatibility Confidence,Local Availability Gap,Sourcing Practicality,Evidence Confidence
0,BYD Seagull,Brake pads,8,4,4.0,3,4.0,4.0
1,BYD Seagull,Brake discs,3,4,4.0,3,4.0,4.0
2,BYD Seagull,Cabin air filters,6,4,3.0,3,4.0,3.0
3,BYD Seagull,Wiper blades,3,4,4.0,3,4.0,4.0
4,BYD Seagull,Headlights,5,4,4.0,3,4.0,4.0
5,BYD Seagull,Tail lights,4,4,4.0,3,4.0,4.0
6,BYD Seagull,Side mirrors,2,4,4.0,3,4.0,4.0
7,BYD Seagull,Bumpers,10,4,4.0,3,4.0,4.0
8,BYD Seagull,Fenders/body panels,11,4,4.0,3,4.0,4.0
9,BYD Seagull,Shock absorbers,7,4,4.0,3,4.0,4.0


Total vehicle-part combinations: 40
Scoring completed successfully.


### 20.4 — Calculate overall category scores

The vehicle–part scores are aggregated by part category. The agreed weights are applied to the average factor scores to produce one overall weighted score for each category. Only combinations with all five factor scores are included, as in the original method.

In [69]:
# Part 20.4: Calculate overall scores for the ten spare-part categories

# Agreed scoring weights
weights = {
    "Market Relevance": 0.35,
    "Compatibility Confidence": 0.25,
    "Local Availability Gap": 0.20,
    "Sourcing Practicality": 0.15,
    "Evidence Confidence": 0.05
}

factor_columns = list(weights.keys())

# Keep only combinations with all five factor scores
complete_combinations = scoring_table.dropna(
    subset=factor_columns
).copy()

# Calculate average factor scores for each spare-part category
category_scores = (
    complete_combinations
    .groupby("part_category")
    .agg(
        vehicles_evaluated=("vehicle", "nunique"),
        evidence_records=("record_count", "sum"),
        **{
            factor: (factor, "mean")
            for factor in factor_columns
        }
    )
    .reset_index()
)

# Calculate the overall weighted score for each category
category_scores["Overall Weighted Score"] = sum(
    category_scores[factor] * weight
    for factor, weight in weights.items()
)

# Round scores for presentation
score_columns = factor_columns + ["Overall Weighted Score"]

category_scores[score_columns] = (
    category_scores[score_columns].round(2)
)

# Display the 10 category scores without ranking
display(category_scores)

# Summary
print("Total spare-part categories:", len(category_scores))
print(
    "Categories evaluated across all four vehicles:",
    (category_scores["vehicles_evaluated"] == 4).sum()
)
print(
    "Categories with fewer than four vehicles:",
    (category_scores["vehicles_evaluated"] < 4).sum()
)
print("Weights total:", sum(weights.values()))

# Export category-level scoring results
category_output = (
    "/content/ev_parts_project/output/EV_Parts_Category_Scores_Working.csv"
)

category_scores.to_csv(category_output, index=False)

print("\nCategory scoring results saved to:", category_output)

,part_category,vehicles_evaluated,evidence_records,Market Relevance,Compatibility Confidence,Local Availability Gap,Sourcing Practicality,Evidence Confidence,Overall Weighted Score
0,Brake discs,3,14,3.67,3.33,3.0,3.33,3.00,3.37
1,Brake pads,4,24,3.75,3.50,3.0,4.00,3.50,3.56
2,Bumpers,4,28,3.75,3.25,3.0,4.00,3.50,3.50
3,Cabin air filters,4,14,3.75,2.75,3.0,4.00,3.25,3.36
4,Fenders/body panels,4,18,3.75,3.00,3.0,4.00,3.25,3.42
5,Headlights,4,19,3.75,3.25,3.0,4.00,3.50,3.50
6,Shock absorbers,4,19,3.75,2.75,3.0,4.00,3.25,3.36
7,Side mirrors,4,15,3.75,3.00,3.0,4.00,3.50,3.44
8,Tail lights,4,14,3.75,3.25,3.0,4.00,3.25,3.49
9,Wiper blades,4,16,3.75,3.00,3.0,4.00,3.50,3.44


Total spare-part categories: 10
Categories evaluated across all four vehicles: 9
Categories with fewer than four vehicles: 1
Weights total: 1.0

Category scoring results saved to: /content/ev_parts_project/output/EV_Parts_Category_Scores_Working.csv


### 20.5 — Perform sensitivity analysis

The baseline weights are compared with four alternative weighting scenarios. This tests whether the category rankings remain stable when greater emphasis is placed on market relevance, compatibility, local availability gap, or sourcing practicality. The factor scoring itself is not changed.

In [70]:
# Part 20.5: Test the sensitivity of the rankings to alternative weights

# Alternative weighting scenarios
scenarios = {
    "Baseline": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.25,
        "Local Availability Gap": 0.20,
        "Sourcing Practicality": 0.15,
        "Evidence Confidence": 0.05
    },

    "Market Focus": {
        "Market Relevance": 0.40,
        "Compatibility Confidence": 0.25,
        "Local Availability Gap": 0.15,
        "Sourcing Practicality": 0.15,
        "Evidence Confidence": 0.05
    },

    "Compatibility Focus": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.30,
        "Local Availability Gap": 0.20,
        "Sourcing Practicality": 0.10,
        "Evidence Confidence": 0.05
    },

    "Availability Focus": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.20,
        "Local Availability Gap": 0.25,
        "Sourcing Practicality": 0.15,
        "Evidence Confidence": 0.05
    },

    "Sourcing Focus": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.25,
        "Local Availability Gap": 0.15,
        "Sourcing Practicality": 0.20,
        "Evidence Confidence": 0.05
    }
}

# Calculate scores under each scenario
sensitivity_results = category_scores[
    ["part_category", "vehicles_evaluated"]
].copy()

for scenario_name, scenario_weights in scenarios.items():

    sensitivity_results[scenario_name] = sum(
        category_scores[factor] * weight
        for factor, weight in scenario_weights.items()
    )

    # Rank categories within each scenario
    sensitivity_results[f"{scenario_name} Rank"] = (
        sensitivity_results[scenario_name]
        .rank(method="min", ascending=False)
    )

# Count how often each category appears in the top five
rank_columns = [
    f"{scenario_name} Rank"
    for scenario_name in scenarios
]

sensitivity_results["Top Five Frequency"] = (
    sensitivity_results[rank_columns].le(5).sum(axis=1)
)

# Display scores and rankings
display(
    sensitivity_results.sort_values(
        "Baseline",
        ascending=False
    ).round(2)
)

# Display top-five stability summary
stability_summary = (
    sensitivity_results[
        ["part_category", "Top Five Frequency"]
    ]
    .sort_values(
        ["Top Five Frequency", "part_category"],
        ascending=[False, True]
    )
)

print("\nTOP-FIVE STABILITY ACROSS SCENARIOS")
display(stability_summary)

# Export results
sensitivity_output = (
    "/content/ev_parts_project/output/EV_Parts_Sensitivity_Analysis.csv"
)

sensitivity_results.to_csv(sensitivity_output, index=False)

print("\nSensitivity analysis saved to:", sensitivity_output)
print("Number of weighting scenarios:", len(scenarios))

,part_category,vehicles_evaluated,Baseline,Baseline Rank,Market Focus,Market Focus Rank,Compatibility Focus,Compatibility Focus Rank,Availability Focus,Availability Focus Rank,Sourcing Focus,Sourcing Focus Rank,Top Five Frequency
1,Brake pads,4,3.56,1.0,3.60,1.0,3.54,1.0,3.54,1.0,3.61,1.0,5
2,Bumpers,4,3.50,2.0,3.54,2.0,3.46,2.0,3.49,2.0,3.55,2.0,5
5,Headlights,4,3.50,2.0,3.54,2.0,3.46,2.0,3.49,2.0,3.55,2.0,5
8,Tail lights,4,3.49,4.0,3.53,4.0,3.45,4.0,3.48,4.0,3.54,4.0,5
9,Wiper blades,4,3.44,5.0,3.48,5.0,3.39,5.0,3.44,5.0,3.49,5.0,5
7,Side mirrors,4,3.44,5.0,3.48,5.0,3.39,5.0,3.44,5.0,3.49,5.0,5
4,Fenders/body panels,4,3.42,7.0,3.46,7.0,3.38,7.0,3.42,7.0,3.48,7.0,0
0,Brake discs,3,3.37,8.0,3.40,10.0,3.37,8.0,3.35,10.0,3.38,10.0,0
3,Cabin air filters,4,3.36,9.0,3.40,8.0,3.30,9.0,3.38,8.0,3.41,8.0,0
6,Shock absorbers,4,3.36,9.0,3.40,8.0,3.30,9.0,3.38,8.0,3.41,8.0,0



TOP-FIVE STABILITY ACROSS SCENARIOS


,part_category,Top Five Frequency
1,Brake pads,5
2,Bumpers,5
5,Headlights,5
7,Side mirrors,5
8,Tail lights,5
9,Wiper blades,5
0,Brake discs,0
3,Cabin air filters,0
4,Fenders/body panels,0
6,Shock absorbers,0



Sensitivity analysis saved to: /content/ev_parts_project/output/EV_Parts_Sensitivity_Analysis.csv
Number of weighting scenarios: 5


### 20.6 — Identify the five highest-priority part categories

The baseline weighted scores are used to identify the five highest-priority categories. The sensitivity results are included to show how consistently each category appears among the top five when the factor weights are varied. The ranking is calculated from the scoring results rather than entered manually.

In [71]:
# Part 20.6: Select the five highest-scoring part categories
final_top_five = (
    category_scores
    .merge(
        sensitivity_results[
            ["part_category", "Top Five Frequency"]
        ],
        on="part_category",
        how="left"
    )
    .sort_values("Overall Weighted Score", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

final_top_five.insert(
    0, "Final Rank", range(1, len(final_top_five) + 1)
)

# Display final recommendations
display(final_top_five[
    [
        "Final Rank",
        "part_category",
        "Overall Weighted Score",
        "vehicles_evaluated",
        "evidence_records",
        "Top Five Frequency"
    ]
])

# Export final recommendations
from pathlib import Path

output_dir = Path(output_dir)

final_top_five.to_csv(
    output_dir / "EV_Parts_Final_Top_Five_Recommendations.csv",
    index=False
)

print("Final top-five recommendations exported successfully.")

,Final Rank,part_category,Overall Weighted Score,vehicles_evaluated,evidence_records,Top Five Frequency
0,1,Brake pads,3.56,4,24,5
1,2,Bumpers,3.50,4,28,5
2,3,Headlights,3.50,4,19,5
3,4,Tail lights,3.49,4,14,5
4,5,Wiper blades,3.44,4,16,5


Final top-five recommendations exported successfully.


## 21. Interpretation of Final Recommendations

The weighted scoring model identified the following five part categories as the highest-priority candidates for potential import:

1. **Brake pads (3.56):** Ranked first, with evidence across all four selected vehicles and consistent top-five performance across all five weighting scenarios.
2. **Bumpers (3.50):** Ranked second, supported by the highest number of evidence records among the recommended categories and consistent performance across all scenarios.
3. **Headlights (3.50):** Ranked third, with evidence across all four vehicles and consistent top-five performance across all scenarios.
4. **Tail lights (3.49):** Ranked fourth, supported by evidence across all four vehicles and stable performance under different weighting scenarios.
5. **Wiper blades (3.44):** Ranked fifth, with evidence across all four vehicles and consistent top-five performance across all five scenarios.

These scores represent the results of my proposed prioritization model. They do not directly measure sales volume or guarantee profitability.

The recommendations provide a starting point for deciding which EV spare-part categories deserve further sourcing and market investigation.

# 22. Final Research Report

## 22.1 Project Overview

This research focused on identifying and prioritizing electric vehicle (EV) spare parts with potential import relevance in Ethiopia. Four EV models were selected: BYD Seagull, BYD Yuan Up, Changan BenBen E-Star, and Toyota bZ4X.

Ten spare-part categories were investigated, covering common maintenance and replacement needs, as well as exterior components.

The main objective was to use available online evidence to develop a structured, evidence-based approach for identifying which spare parts may deserve greater import priority.

## 22.2 Data Collection and Preparation

Publicly available online information was collected in two research phases. The second phase supplemented the initial dataset, particularly where evidence coverage was limited.

The final dataset contained 197 records, including vehicle-specific spare-part evidence and contextual information about the Ethiopian EV market. The collected information included supplier and marketplace listings, product details, compatibility information, prices, and general market context.

The raw dataset was preserved, while a separate working dataset was prepared for analysis. Data preparation involved standardizing relevant fields, checking records, handling missing values, extracting numerical price information, and converting eligible currencies into Ethiopian Birr using the selected reference exchange rates.

Original values were retained to support traceability. Information that could not be reliably interpreted or converted was not replaced with guessed values.

## 22.3 Prioritization Method

A weighted scoring method was developed to compare the spare-part categories. Five factors were considered:

- Market Relevance – 35%
- Compatibility Confidence – 25%
- Local Availability Gap – 20%
- Sourcing Practicality – 15%
- Evidence Confidence – 5%

Each factor contributes to the overall score according to its assigned weight. The method combines market-related considerations with compatibility, sourcing, and evidence quality.

Scores were first calculated for individual vehicle–part combinations and then aggregated to compare the ten spare-part categories.

A sensitivity analysis was also conducted using five weighting scenarios: baseline, market focus, compatibility focus, availability focus, and sourcing focus. This helped assess whether the category rankings remained consistent when the importance of the evaluation factors changed.

The scores represent a comparative prioritization model based on the available research. They should not be interpreted as confirmed sales demand or guaranteed import profitability.

## 22.4 Results and Recommendations

The analysis produced scores for the investigated vehicle–part combinations and aggregated them to rank the ten spare-part categories.

The five highest-ranked categories under the baseline weighting scenario were:

1. **Brake pads (3.56):** Ranked first, with evidence across all four selected vehicles.
2. **Bumpers (3.50):** Ranked second, with the highest number of evidence records among the five recommended categories.
3. **Headlights (3.50):** Ranked third, with evidence across all four selected vehicles.
4. **Tail lights (3.49):** Ranked fourth, supported by evidence across all four vehicles.
5. **Wiper blades (3.44):** Ranked fifth, with evidence across all four selected vehicles.

All five categories appeared among the top five in each of the five weighting scenarios, indicating consistent performance under the tested changes in factor importance.

These recommendations identify the strongest priorities according to the selected criteria and available evidence. They provide a starting point for further supplier investigation and market validation before making actual import decisions.

## 22.5 Limitations and Unavailable Information

Several limitations should be considered when interpreting the findings:

- **Limited local market data:** Reliable Ethiopian item-level prices, stock information, and sales figures were not available for every spare-part category.
- **Incomplete compatibility evidence:** Some listings did not provide sufficient part numbers or vehicle-specific fitment details to confirm compatibility.
- **Uneven evidence coverage:** The quantity and quality of available records varied across the four selected vehicle models.
- **Price limitations:** Some prices were missing, listed as price-on-request, or provided in currencies that could not be reliably converted.
- **Source access limitations:** Some observations were based on search-result excerpts rather than complete webpage content.
- **No physical verification:** Supplier stock, actual product condition, and local market demand were not independently confirmed through physical inspections or interviews.
- **Limited local availability evidence:** The Local Availability Gap score remained 3 across all 40 vehicle–part combinations because the collected sources did not establish either verified local availability or a confirmed model-specific supply shortage.

International product listings can provide useful sourcing information, but they do not prove local Ethiopian availability or demand. Similarly, vehicle listings do not establish actual registration or sales numbers.

Therefore, the results should be treated as research-based estimates rather than confirmed market conditions.

## 22.6 Conclusion

This project developed a structured process for researching, preparing, and analyzing EV spare-part information for the Ethiopian market.

By combining publicly available evidence, data cleaning, compatibility assessment, and a weighted scoring method, the research produced a comparative list of spare-part import priorities.

The final analysis identified brake pads, bumpers, headlights, tail lights, and wiper blades as the five highest-priority categories under the proposed model. Their consistent performance across the five weighting scenarios provides additional support for their selection.

Although further local market research and supplier verification are necessary, the findings provide a useful foundation for investigating potential EV spare-part import opportunities in Ethiopia.